## **C2 — MedMamba-T + EDL + HAAL**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [ ]:
!pip -q install packaging ninja einops "timm>=1.0" scikit-learn pandas numpy matplotlib pillow tqdm thop scipy requests
!pip -q install --no-build-isolation "causal-conv1d>=1.4.0" "mamba-ssm>=2.2.0"



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 17.1 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 56.3 MB/s eta 0:00:00


In [ ]:
import os
import gc
import ast
import json
import math
import time
import random
import warnings
import types
import copy
import hashlib
from pathlib import Path
from collections import Counter

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.preprocessing import label_binarize
from scipy.stats import wilcoxon, pearsonr
from tqdm.auto import tqdm
from thop import profile

warnings.filterwarnings("ignore")



1. Google Drive folder structure



In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/LIDC_Thesis")
LIDC_ROOT = PROJECT_ROOT / "LIDC"
PROCESSED_ROOT = LIDC_ROOT / "processed"
PATCH_ROOT = PROCESSED_ROOT / "patches"
METADATA_ROOT = PROCESSED_ROOT / "metadata"

C1_ROOT = PROJECT_ROOT / "experiments" / "C1_ConvNeXtV2T_EDL_HAAL"
C2_ROOT = PROJECT_ROOT / "experiments" / "C2_MedMambaT_HAAL"

NOTEBOOK_ROOT = C2_ROOT / "notebooks"
CONFIG_ROOT = C2_ROOT / "configs"
SPLIT_ROOT = C2_ROOT / "splits"
CHECKPOINT_ROOT = C2_ROOT / "checkpoints"
PREDICTION_ROOT = C2_ROOT / "predictions"
METRICS_ROOT = C2_ROOT / "metrics"
GRADCAM_ROOT = C2_ROOT / "gradcam"
ESM_ROOT = C2_ROOT / "esm"
PLOT_ROOT = C2_ROOT / "plots"
LOG_ROOT = C2_ROOT / "logs"
REPORT_ROOT = C2_ROOT / "reports"
ABLATION_ROOT = C2_ROOT / "ablations"

MEDMAMBA_ROOT = PROJECT_ROOT / "models" / "MedMamba"
MEDMAMBA_SOURCE_ROOT = MEDMAMBA_ROOT / "source"
MEDMAMBA_WEIGHT_ROOT = MEDMAMBA_ROOT / "weights"

for directory in [
    NOTEBOOK_ROOT, CONFIG_ROOT, SPLIT_ROOT, CHECKPOINT_ROOT,
    PREDICTION_ROOT, METRICS_ROOT, GRADCAM_ROOT, ESM_ROOT,
    PLOT_ROOT, LOG_ROOT, REPORT_ROOT, ABLATION_ROOT,
    MEDMAMBA_SOURCE_ROOT, MEDMAMBA_WEIGHT_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

METADATA_CSV = METADATA_ROOT / "lidc_patches_metadata.csv"
# Robust C1 config/provenance discovery.
# Prefer the actual experiment path already used in your successful C1 inheritance,
C1_CONFIG_CANDIDATES = [
    Path("/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_EDL_HAAL/configs/C1_config.json"),
    Path("/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_EDL_HAAL/configs/run_provenance.json"),
    Path("/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_EDL_HAAL/run_provenance.json"),
    Path("/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_HAAL/configs/C1_config.json"),
    Path("/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_HAAL/configs/run_provenance.json"),
    Path("/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_HAAL/run_provenance.json"),
]

# Also search the experiments tree so minor folder-name changes do not break C2.
C1_SEARCH_ROOT = Path("/content/drive/MyDrive/LIDC_Thesis/experiments")
if C1_SEARCH_ROOT.exists():
    for pattern in [
        "C1*/configs/C1_config.json",
        "C1*/configs/run_provenance.json",
        "C1*/run_provenance.json",
    ]:
        C1_CONFIG_CANDIDATES.extend(sorted(C1_SEARCH_ROOT.glob(pattern)))

# De-duplicate while preserving priority.
_seen = set()
C1_CONFIG_CANDIDATES = [
    p for p in C1_CONFIG_CANDIDATES
    if not (str(p) in _seen or _seen.add(str(p)))
]

C1_CONFIG_PATH = next((p for p in C1_CONFIG_CANDIDATES if p.exists()), None)

if C1_CONFIG_PATH is None:
    checked = "\n".join(f"  - {p}" for p in C1_CONFIG_CANDIDATES)
    raise FileNotFoundError(
        "Could not find the final C1 config/provenance automatically.\n"
        "Checked:\n" + checked + "\n\n"
        "Expected the C1 experiment under "
        "/content/drive/MyDrive/LIDC_Thesis/experiments/."
    )

print("Resolved C1 config/provenance:", C1_CONFIG_PATH)
MEDMAMBA_WEIGHTS_PATH = MEDMAMBA_WEIGHT_ROOT / "medmamba_t.pth"

print("C2 root:", C2_ROOT)
print("Optional MedMamba weights:", MEDMAMBA_WEIGHTS_PATH)


Resolved C1 config/provenance: /content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_EDL_HAAL/configs/C1_config.json
C2 root: /content/drive/MyDrive/LIDC_Thesis/experiments/C2_MedMambaT_HAAL
Optional MedMamba weights: /content/drive/MyDrive/LIDC_Thesis/models/MedMamba/weights/medmamba_t.pth


2. Configuration and reproducibility



In [ ]:
DEFAULT_C1 = {
    "prior_c": 1.0,
    "lambda_max": 5.0,
    "t_warmup": 0,
    "objective_mode": "reader_distribution",
    "evidence_reg_max": 0.01,
    "evidence_reg_warmup": 10,
}

# Final C2 should inherit the frozen C1 objective/settings so C1 vs C2 isolates backbone.
REQUIRE_C1_CONFIG = True

def load_inherited_c1():
    if C1_CONFIG_PATH is None or not C1_CONFIG_PATH.exists():
        if REQUIRE_C1_CONFIG:
            raise FileNotFoundError(
                f"C1 config/provenance not found: {C1_CONFIG_PATH}. "
                "Point C1_CONFIG_PATH to the final C1 config or run_provenance.json."
            )
        return DEFAULT_C1.copy(), "provisional_defaults_C1_config_missing"

    c1_cfg = json.loads(C1_CONFIG_PATH.read_text(encoding="utf-8-sig"))

    # Supports final run_provenance.json schema:
    # {"objective_mode": ..., "selected_hyperparameters": {"lambda_max": ..., "t_warmup": ...}, ...}
    if "selected_hyperparameters" in c1_cfg:
        selected = c1_cfg["selected_hyperparameters"]
        values = {
            "prior_c": float(c1_cfg.get("prior_c", c1_cfg.get("haal_prior_c_default", 1.0))),
            "lambda_max": float(selected["lambda_max"]),
            "t_warmup": int(selected.get("t_warmup", selected.get("T_warmup", 0))),
            "objective_mode": str(c1_cfg.get("objective_mode", "reader_distribution")),
            "evidence_reg_max": float(c1_cfg.get("evidence_reg_max", 0.01)),
            "evidence_reg_warmup": int(c1_cfg.get("evidence_reg_warmup", 10)),
        }
        return values, f"inherited_from:{C1_CONFIG_PATH}"

    # Supports C1_config.json variants (new and legacy).
    prior = c1_cfg.get("prior_c",
             c1_cfg.get("prior_weight_C",
             c1_cfg.get("haal_prior_c_default", 1.0)))
    lam = c1_cfg.get("lambda_max",
           c1_cfg.get("haal_lambda_max_default", None))
    warm = c1_cfg.get("t_warmup",
            c1_cfg.get("T_warmup",
            c1_cfg.get("haal_t_warmup_default", None)))

    if lam is None or warm is None:
        raise KeyError(
            "Could not resolve final C1 lambda_max/t_warmup from the supplied JSON. "
            f"Found top-level keys: {sorted(c1_cfg)}"
        )

    values = {
        "prior_c": float(prior),
        "lambda_max": float(lam),
        "t_warmup": int(warm),
        "objective_mode": str(c1_cfg.get("objective_mode", "reader_distribution")),
        "evidence_reg_max": float(c1_cfg.get("evidence_reg_max", 0.01)),
        "evidence_reg_warmup": int(c1_cfg.get("evidence_reg_warmup", 10)),
    }
    return values, f"inherited_from:{C1_CONFIG_PATH}"

C1_INHERITED, C1_SOURCE = load_inherited_c1()

print("C1 source:", C1_SOURCE)
print("C1 inherited settings:", C1_INHERITED)

CONFIG = {
    "config_id": "C2",
    "backbone": "MedMamba-T",
    "uq_method": "Reader-aligned EDL",
    "loss": "soft-reader EDL CE + lambda(t)*KL(q||p) + evidence regularization",
    "objective_mode": C1_INHERITED["objective_mode"],
    "num_classes": 3,
    "class_names": ["benign", "indeterminate", "malignant"],

    "medmamba_depths": [2, 2, 4, 2],
    "medmamba_dims": [96, 192, 384, 768],
    "d_state": 16,
    "drop_path_rate": 0.1,
    "use_checkpoint": False,

    "seed": 42,
    "use_precomputed_folds": True,
    "fold_column": "fold",
    "n_splits": 5,
    "validation_fraction_of_training_patients": 0.10,

    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,

    "max_epochs": 50,
    "early_stopping_patience": 10,

    "learning_rate": 1e-4,
    "eta_min": 1e-6,
    "weight_decay": 0.01,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,

    # Canonical final-C1 names
    "prior_c": C1_INHERITED["prior_c"],
    "lambda_max": C1_INHERITED["lambda_max"],
    "t_warmup": C1_INHERITED["t_warmup"],
    "evidence_reg_max": C1_INHERITED["evidence_reg_max"],
    "evidence_reg_warmup": C1_INHERITED["evidence_reg_warmup"],
    "c1_source": C1_SOURCE,

    # Compatibility aliases retained because later v1.1 cells use these names.
    "prior_weight_C": C1_INHERITED["prior_c"],
    "T_warmup": C1_INHERITED["t_warmup"],
    "haal_source": C1_SOURCE,

    "lambda_kl_for_ablation": 1.0,
    "epsilon": 1e-8,
    "ece_bins": 15,
    "random_rejection_repeats": 50,
    "saliency_max_examples_per_fold": 30,
    "xai_metric_steps": 20,

    "pilot_max_patients": None,

    # Legacy sensitivity/ablation switches are retained but OFF by default.
    "ablation_lambda_max_values": [0.0, 0.1, 0.5, 1.0, 5.0],
    "ablation_T_warmup_values": [0, 5, 10, 20],
    "ablation_prior_C_values": [0.5, 1.0, 2.0],
    "ablation_loss_modes": ["reader_distribution", "legacy_v_haal", "ece_only", "ece_plus_kl"],
    "medmamba_hook_modes": ["pre", "post", "internal"],

    "augmentation": {
        "horizontal_flip_p": 0.5,
        "vertical_flip_p": 0.5,
        "rotation_degrees": 15,
        "translation_fraction": 0.10,
        "elastic_deformation": False,
        "intensity_jitter": False,
    },
}

with open(CONFIG_ROOT / "C2_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("C1 settings source:", CONFIG["c1_source"])
print("Objective:", CONFIG["objective_mode"])
print("C / lambda_max / t_warmup:",
      CONFIG["prior_c"], CONFIG["lambda_max"], CONFIG["t_warmup"])
print("Evidence reg max / warmup:",
      CONFIG["evidence_reg_max"], CONFIG["evidence_reg_warmup"])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

with open(CONFIG_ROOT / "random_seeds.json", "w") as f:
    json.dump({
        "global_seed": CONFIG["seed"],
        "libraries": ["python random", "numpy", "torch", "cuda"],
        "dataloader_generator_seed": CONFIG["seed"],
    }, f, indent=2)


C1 source: inherited_from:/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_EDL_HAAL/configs/C1_config.json
C1 inherited settings: {'prior_c': 1.0, 'lambda_max': 5.0, 't_warmup': 0, 'objective_mode': 'reader_distribution', 'evidence_reg_max': 0.01, 'evidence_reg_warmup': 10}
Device: cuda
C1 settings source: inherited_from:/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_EDL_HAAL/configs/C1_config.json
Objective: reader_distribution
C / lambda_max / t_warmup: 1.0 5.0 0
Evidence reg max / warmup: 0.01 10


3. Load official MedMamba source
The official source is downloaded at runtime. The demonstration code at the
bottom of the repository file is removed before execution so importing the
architecture does not create unrelated CUDA models.



In [ ]:
OFFICIAL_MEDMAMBA_URL = (
    "https://raw.githubusercontent.com/"
    "YubiaoYue/MedMamba/main/MedMamba.py"
)

OFFICIAL_SOURCE_PATH = MEDMAMBA_SOURCE_ROOT / "MedMamba_official.py"

response = requests.get(OFFICIAL_MEDMAMBA_URL, timeout=60)
response.raise_for_status()
OFFICIAL_SOURCE_PATH.write_text(response.text)

medmamba_source = response.text

for marker in [
    "\nmedmamba_t = VSSM(",
    "\nmedmamba_s = VSSM(",
    "\ndata = torch.randn(",
]:
    if marker in medmamba_source:
        medmamba_source = medmamba_source.split(marker)[0]
        break

MEDMAMBA_NS = {
    "__name__": "medmamba_official_runtime",
    "__file__": str(OFFICIAL_SOURCE_PATH),
}

exec(
    compile(
        medmamba_source,
        str(OFFICIAL_SOURCE_PATH),
        "exec",
    ),
    MEDMAMBA_NS,
)

VSSM = MEDMAMBA_NS["VSSM"]
SS2D = MEDMAMBA_NS["SS2D"]
SS_Conv_SSM = MEDMAMBA_NS["SS_Conv_SSM"]

print("Official MedMamba VSSM loaded.")

# The scan kernel comes from mamba_ssm. The official file hides an import failure with try/except,
# so check it here instead of failing later inside the first forward pass.
if "selective_scan_fn" not in MEDMAMBA_NS:
    raise RuntimeError(
        "mamba_ssm selective_scan_fn was not imported. Re-run the pip cell and check the install log "
        "(needs a CUDA GPU runtime and a matching torch build)."
    )
if not torch.cuda.is_available():
    raise RuntimeError("MedMamba needs a CUDA GPU. Change the Colab runtime type to A100 or V100.")

MEDMAMBA_SHA256 = hashlib.sha256(response.content).hexdigest()
print("MedMamba_official.py sha256:", MEDMAMBA_SHA256)


Official MedMamba VSSM loaded.
MedMamba_official.py sha256: 52c0c3295de5b7ba943c1a4d9ce9c9fa4a4b440e7c7c86d0de1fe2ab53f64780


4. Metadata loading / reconstruction



In [ ]:
LABEL_MAP = {"benign": 0, "indeterminate": 1, "malignant": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

COLUMN_ALIASES = {
    "patch_path": "image_path",
    "label_id": "label",
    "label_name": "class_name",
}

def infer_patient_id(fp, class_dir):
    rel = fp.relative_to(class_dir)
    if len(rel.parts) >= 2:
        return str(rel.parts[0])
    stem = fp.stem
    if stem.startswith("LIDC-IDRI-") and len(stem) >= 14:
        return stem[:14]
    return stem.split("_")[0]

def rebuild_metadata_from_patch_folders(root):
    rows = []
    extensions = {".npy", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
    for class_name, label in LABEL_MAP.items():
        class_dir = root / class_name
        if not class_dir.exists():
            continue
        for fp in sorted(class_dir.rglob("*")):
            if not fp.is_file() or fp.suffix.lower() not in extensions:
                continue
            rows.append({
                "image_path": str(fp),
                "patient_id": infer_patient_id(fp, class_dir),
                "label": int(label),
                "class_name": class_name,
            })
    print("WARNING: metadata CSV not found, rebuilt from folders. Variance V and fold ids will be missing.")
    return pd.DataFrame(rows)

def find_metadata_csv():
    candidates = [METADATA_CSV, PROCESSED_ROOT / "lidc_patches_metadata.csv", LIDC_ROOT / "lidc_patches_metadata.csv"]
    for c in candidates:
        if c.exists():
            return c
    found = list(PROCESSED_ROOT.rglob("lidc_patches_metadata.csv"))
    return found[0] if found else None

def resolve_patch_path(p):
    p = Path(str(p))
    if p.exists():
        return str(p)
    for base in (PROCESSED_ROOT, LIDC_ROOT, PROJECT_ROOT):
        if (base / p).exists():
            return str(base / p)
    return str(p)

csv_path = find_metadata_csv()
if csv_path is not None:
    metadata = pd.read_csv(csv_path)
    print("Metadata CSV:", csv_path)
    print("Columns:", list(metadata.columns))
    for old, new in COLUMN_ALIASES.items():
        if old in metadata.columns and new not in metadata.columns:
            metadata = metadata.rename(columns={old: new})
else:
    metadata = rebuild_metadata_from_patch_folders(PATCH_ROOT)

if len(metadata) == 0:
    raise RuntimeError("No patch metadata or patches found.")

if "patient_id" not in metadata.columns and "image_path" in metadata.columns:
    metadata["patient_id"] = metadata["image_path"].map(
        lambda p: infer_patient_id(Path(str(p)), Path(str(p)).parent)
    )

for col in ["image_path", "patient_id", "label"]:
    if col not in metadata.columns:
        raise ValueError(f"Missing metadata column: {col}. Available: {list(metadata.columns)}")

metadata["image_path"] = metadata["image_path"].map(resolve_patch_path)
metadata["patient_id"] = metadata["patient_id"].astype(str)
metadata["label"] = metadata["label"].astype(int)

if "class_name" not in metadata.columns:
    metadata["class_name"] = metadata["label"].map(INV_LABEL_MAP)
metadata["class_name"] = metadata["class_name"].astype(str).str.strip().str.lower()

# The class order must be identical to the preprocessing; a swapped order would be silent otherwise.
if not (metadata["class_name"].map(LABEL_MAP) == metadata["label"]).all():
    print(pd.crosstab(metadata["label"], metadata["class_name"]))
    raise ValueError("label ids do not match LABEL_MAP (benign=0, indeterminate=1, malignant=2).")

missing_files = ~metadata["image_path"].map(lambda p: Path(p).exists())
if missing_files.any():
    metadata.loc[missing_files, ["image_path"]].to_csv(LOG_ROOT / "missing_patch_files.csv", index=False)
    print(f"WARNING: {int(missing_files.sum())} patch files not found, see logs/missing_patch_files.csv")
metadata = metadata[~missing_files].drop_duplicates("image_path").reset_index(drop=True)

if CONFIG["pilot_max_patients"] is not None:
    selected_patients = sorted(metadata["patient_id"].unique())[:CONFIG["pilot_max_patients"]]
    metadata = metadata[metadata["patient_id"].isin(selected_patients)].reset_index(drop=True)
    print("WARNING: pilot subset active. Precomputed folds will be unbalanced and differ from the other configs.")

if "mask_path" not in metadata.columns:
    print("NOTE: no mask_path column. Pointing Game and Boundary Distance will be skipped "
          "until the shared contour-mask export is linked (Plan 8.3).")

print(metadata.to_string())
print("Patches:", len(metadata))
print("Patients:", metadata["patient_id"].nunique())
display(metadata["class_name"].value_counts())


Metadata CSV: /content/drive/MyDrive/LIDC_Thesis/LIDC/processed/metadata/lidc_patches_metadata.csv
Columns: ['nodule_id', 'patient_id', 'scan_db_id', 'series_instance_uid', 'label_name', 'label_id', 'malignancy_scores', 'malignancy_median', 'n_radiologists', 'diameter_mm', 'variance_raw', 'variance_normalized', 'alignment_mask', 'patch_path', 'central_slice_k', 'fold']
NOTE: no mask_path column. Pointing Game and Boundary Distance will be skipped until the shared contour-mask export is linked (Plan 8.3).
                  nodule_id      patient_id  scan_db_id                                               series_instance_uid     class_name  label malignancy_scores  malignancy_median  n_radiologists  diameter_mm  variance_raw  variance_normalized  alignment_mask                                                                                           image_path  central_slice_k  fold
0   LIDC-IDRI-0001_S12_N000  LIDC-IDRI-0001          12  1.3.6.1.4.1.14519.5.2.1.6279.6001.17904937363643

,count
class_name,
malignant,44
indeterminate,32
benign,20


5. HAAL variance target V
C2 cannot be trained correctly from class labels alone. V is either loaded
from metadata or computed from individual radiologist malignancy scores.



In [ ]:
def parse_score_list(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        raw = list(value)

    elif pd.isna(value):
        return []

    else:
        text = str(value).strip()

        if not text:
            return []

        try:
            parsed = json.loads(text)
        except Exception:
            try:
                parsed = ast.literal_eval(text)
            except Exception:
                parsed = [
                    x.strip()
                    for x in text.replace(";", ",").split(",")
                    if x.strip()
                ]

        raw = (
            list(parsed)
            if isinstance(parsed, (list, tuple, np.ndarray))
            else [parsed]
        )

    scores = []

    for value in raw:
        try:
            score = float(value)
        except Exception:
            continue

        if 1 <= score <= 5:
            scores.append(score)

    return scores

def variance_target_from_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)

    if len(scores) < 3:
        return np.nan

    return float(
        np.clip(
            np.var(scores, ddof=0) / 4.0,
            0.0,
            1.0,
        )
    )

def attach_variance_target(df):
    out = df.copy()

    for candidate in [
        "variance_target",
        "V",
        "normalized_variance",
        "annotator_variance_normalized",
        "variance_norm",
        "haal_target",
    ]:
        if candidate in out.columns:
            values = pd.to_numeric(out[candidate], errors="coerce")

            if values.notna().any():
                out["variance_target"] = values.clip(0, 1)
                return out, f"existing:{candidate}"

    for candidate in [
        "radiologist_scores",
        "malignancy_scores",
        "reader_scores",
        "annotation_scores",
    ]:
        if candidate in out.columns:
            score_lists = out[candidate].map(parse_score_list)
            values = score_lists.map(variance_target_from_scores)

            if values.notna().any():
                out["variance_target"] = values
                out["radiologist_score_count"] = score_lists.map(len)
                return out, f"computed_from:{candidate}"

    score_columns = [
        c for c in out.columns
        if (
            c.lower().startswith("malignancy_")
            or c.lower().startswith("reader_score_")
            or c.lower().startswith("radiologist_score_")
        )
    ]

    if score_columns:
        def row_target(row):
            scores = []

            for col in score_columns:
                try:
                    score = float(row[col])
                except Exception:
                    continue

                if 1 <= score <= 5:
                    scores.append(score)

            return variance_target_from_scores(scores)

        out["variance_target"] = out.apply(row_target, axis=1)

        if out["variance_target"].notna().any():
            return out, "computed_from_columns:" + ",".join(score_columns)

    return out, None

metadata, variance_source = attach_variance_target(metadata)
print(metadata["variance_target"].describe())
print(metadata.to_string())
print(variance_source)

if variance_source is None:
    if CONFIG["objective_mode"] == "reader_distribution":
        metadata["variance_target"] = 0.0
        metadata["alignment_mask"] = 0
        variance_source = "not_required_for_reader_distribution"
        alignment_mask_source = "disabled_for_main_reader_distribution"
        print(
            "Variance target V not found: OK for main reader_distribution training. "
            "Legacy V-HAAL ablation will be unavailable unless V is supplied."
        )
    else:
        diagnostic = METADATA_ROOT / "C2_REQUIRED_VARIANCE_TARGET.txt"
        diagnostic.write_text(
            "Legacy V-HAAL requires normalized annotator-disagreement V for every patch.\n"
            "V = population variance of >=3 malignancy scores / 4.\n"
            "Provide variance_target/V or individual radiologist malignancy scores.\n"
        )
        raise RuntimeError(
            "This objective requires HAAL variance target V. "
            f"See: {diagnostic}"
        )
else:
    metadata["variance_target"] = pd.to_numeric(
        metadata["variance_target"], errors="coerce"
    )

    bad_v = (
        metadata["variance_target"].isna()
        | (metadata["variance_target"] < 0)
        | (metadata["variance_target"] > 1)
    )

    if bad_v.any():
        if CONFIG["objective_mode"] == "reader_distribution":

            metadata.loc[bad_v, "variance_target"] = 0.0
            print(
                f"Warning: {int(bad_v.sum())} invalid/missing V values were replaced "
                "with 0 because V is not used by reader_distribution training."
            )
        else:
            metadata.loc[
                bad_v, ["image_path", "patient_id", "variance_target"]
            ].to_csv(LOG_ROOT / "invalid_variance_targets.csv", index=False)
            raise RuntimeError(
                f"{int(bad_v.sum())} invalid variance targets found."
            )

    if "alignment_mask" in metadata.columns:
        metadata["alignment_mask"] = (
            pd.to_numeric(metadata["alignment_mask"], errors="coerce")
            .fillna(0).astype(int)
        )
        alignment_mask_source = "existing:alignment_mask"
    else:
        metadata["alignment_mask"] = (
            metadata["variance_target"] > 0
        ).astype(int)
        alignment_mask_source = "derived:variance_target>0"

print("alignment_mask source:", alignment_mask_source)
print("Variance source:", variance_source)


if variance_source != "not_required_for_reader_distribution":
    plt.figure(figsize=(7,4))
    plt.hist(metadata["variance_target"], bins=15)
    plt.xlabel("Normalized annotator variance V")
    plt.ylabel("Patch count")
    plt.title("C2 variance-target distribution (legacy V-HAAL diagnostic)")
    plt.tight_layout()
    plt.savefig(PLOT_ROOT / "variance_target_distribution.png", dpi=160)
    plt.show()

metadata.to_csv(
    METADATA_ROOT / "C2_effective_metadata.csv",
    index=False,
)

# ------------------------------------------------------------------
# Final-C1 reader-distribution target q
# Ratings 1-2 -> benign, 3 -> indeterminate, 4-5 -> malignant.
# Prefer existing q_* columns; otherwise reconstruct q from raw scores.
# ------------------------------------------------------------------
Q_COLS = ["q_benign", "q_indeterminate", "q_malignant"]

def scores_to_reader_distribution(scores):
    scores = parse_score_list(scores)
    if len(scores) < 1:
        return [np.nan, np.nan, np.nan]
    counts = np.zeros(3, dtype=np.float64)
    for s in scores:
        if s <= 2:
            counts[0] += 1
        elif s == 3:
            counts[1] += 1
        else:
            counts[2] += 1
    return (counts / counts.sum()).tolist()

def attach_reader_distribution(df):
    out = df.copy()

    if all(c in out.columns for c in Q_COLS):
        q = out[Q_COLS].apply(pd.to_numeric, errors="coerce")
        row_sum = q.sum(axis=1)
        valid = q.notna().all(axis=1) & (row_sum > 0)
        if valid.all():
            out[Q_COLS] = q.div(row_sum, axis=0)
            return out, "existing:q_columns"

    for candidate in [
        "radiologist_scores", "malignancy_scores",
        "reader_scores", "annotation_scores"
    ]:
        if candidate in out.columns:
            q = out[candidate].map(scores_to_reader_distribution)
            qdf = pd.DataFrame(q.tolist(), columns=Q_COLS, index=out.index)
            if qdf.notna().all(axis=1).all():
                out[Q_COLS] = qdf
                return out, f"computed_from:{candidate}"

    score_columns = [
        c for c in out.columns
        if c.lower().startswith(("malignancy_", "reader_score_", "radiologist_score_"))
    ]
    if score_columns:
        def row_scores(row):
            vals = []
            for c in score_columns:
                try:
                    v = float(row[c])
                    if 1 <= v <= 5:
                        vals.append(v)
                except Exception:
                    pass
            return vals
        q = out.apply(lambda r: scores_to_reader_distribution(row_scores(r)), axis=1)
        qdf = pd.DataFrame(q.tolist(), columns=Q_COLS, index=out.index)
        if qdf.notna().all(axis=1).all():
            out[Q_COLS] = qdf
            return out, "computed_from_score_columns"

    return out, None

metadata, reader_distribution_source = attach_reader_distribution(metadata)

if CONFIG["objective_mode"] == "reader_distribution":
    if reader_distribution_source is None:
        raise RuntimeError(
            "Final C1 uses objective_mode='reader_distribution', but C2 could not obtain "
            "q_benign/q_indeterminate/q_malignant or reconstruct them from raw radiologist "
            "malignancy scores. Do not fall back to hard labels for the C1-vs-C2 comparison."
        )

    qmat = metadata[Q_COLS].to_numpy(dtype=np.float64)
    if (qmat < -1e-8).any() or (qmat > 1 + 1e-8).any():
        raise RuntimeError("Invalid reader-distribution probabilities outside [0,1].")
    if not np.allclose(qmat.sum(axis=1), 1.0, atol=1e-6):
        raise RuntimeError("Reader-distribution q rows do not sum to 1.")

    # Normalized reader entropy retained for diagnostics.
    metadata["reader_vote_entropy_normalized"] = (
        -(qmat * np.log(qmat + CONFIG["epsilon"])).sum(axis=1)
        / math.log(CONFIG["num_classes"])
    )

print("Reader-distribution source:", reader_distribution_source)
metadata.to_csv(METADATA_ROOT / "C2_effective_metadata.csv", index=False)


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: variance_target, dtype: float64
                  nodule_id      patient_id  scan_db_id                                               series_instance_uid     class_name  label malignancy_scores  malignancy_median  n_radiologists  diameter_mm  variance_raw  variance_normalized  alignment_mask                                                                                           image_path  central_slice_k  fold  variance_target
0   LIDC-IDRI-0001_S12_N000  LIDC-IDRI-0001          12  1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636438705059720603192      malignant      2           5|5|5|4                5.0               4    32.755812      0.187500             0.046875               1      /content/drive/MyDrive/LIDC_Thesis/LIDC/processed/patches/malignant/LIDC-IDRI-0001_S12_N000.npy               89     1              NaN
1   LIDC-IDRI-0003_S14_N001  LIDC-IDRI-0003          1

6. Integrity checks and patient-level five-fold splits



In [ ]:
def check_patch(path):
    # returns None when the patch is fine, otherwise a short reason
    try:
        if str(path).lower().endswith(".npy"):
            arr = np.load(path)
            if arr.ndim == 3 and arr.shape[0] == 1:
                arr = arr[0]
            if arr.ndim != 2:
                return f"unexpected shape {arr.shape}"
            if not np.isfinite(arr).all():
                return "non-finite values"
            if arr.min() < -1e-4 or arr.max() > 1 + 1e-4:
                return f"values outside [0,1]: {arr.min():.3f}..{arr.max():.3f}"
            return None
        with Image.open(path) as image:
            image.verify()
        return None
    except Exception as exc:
        return repr(exc)

bad_images = []
for path in tqdm(metadata["image_path"], desc="Checking patches"):
    reason = check_patch(path)
    if reason is not None:
        bad_images.append({"image_path": path, "error": reason})

if bad_images:
    pd.DataFrame(bad_images).to_csv(LOG_ROOT / "unreadable_images.csv", index=False)
    bad_set = {x["image_path"] for x in bad_images}
    metadata = metadata[~metadata["image_path"].isin(bad_set)].reset_index(drop=True)
    print(f"WARNING: removed {len(bad_set)} unreadable patches, see logs/unreadable_images.csv")

if metadata["patient_id"].nunique() < CONFIG["n_splits"]:
    raise RuntimeError("Not enough patients for five-fold CV.")

def dominant_patient_label(df):
    return (
        df.groupby("patient_id")["label"]
        .agg(lambda s: int(s.value_counts().index[0]))
    )

def make_outer_splits(df):
    X = np.zeros((len(df), 1))
    y = df["label"].values
    groups = df["patient_id"].values

    try:
        splitter = StratifiedGroupKFold(
            n_splits=CONFIG["n_splits"],
            shuffle=True,
            random_state=CONFIG["seed"],
        )

        splits = list(
            splitter.split(X, y, groups)
        )

        method = "StratifiedGroupKFold"

    except Exception as exc:
        print("Stratified split fallback:", repr(exc))

        splitter = GroupKFold(
            n_splits=CONFIG["n_splits"]
        )

        splits = list(
            splitter.split(X, y, groups)
        )

        method = "GroupKFold"

    return splits, method

def make_train_val_split(trainval_df, fold):
    patient_labels = dominant_patient_label(trainval_df)

    patients = patient_labels.index.to_numpy()
    labels = patient_labels.values

    n_val = max(
        1,
        int(
            round(
                len(patients)
                * CONFIG["validation_fraction_of_training_patients"]
            )
        ),
    )

    n_val = min(n_val, len(patients)-1)

    counts = Counter(labels)

    can_stratify = (
        len(counts) >= 2
        and min(counts.values()) >= 2
        and n_val >= len(counts)
    )

    if can_stratify:
        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=n_val,
            random_state=CONFIG["seed"] + fold,
        )

        train_idx, val_idx = next(
            splitter.split(patients, labels)
        )

        train_patients = set(patients[train_idx])
        val_patients = set(patients[val_idx])

        method = "StratifiedShuffleSplit(patient)"

    else:
        rng = np.random.RandomState(CONFIG["seed"] + fold)
        shuffled = patients.copy()
        rng.shuffle(shuffled)

        val_patients = set(shuffled[:n_val])
        train_patients = set(shuffled[n_val:])

        method = "RandomPatientSplit(fallback)"

    train_df = trainval_df[
        trainval_df["patient_id"].isin(train_patients)
    ].copy()

    val_df = trainval_df[
        trainval_df["patient_id"].isin(val_patients)
    ].copy()

    return train_df, val_df, method


def build_outer_splits(df):
    # Preferred: the fold column from preprocessing, so C2 tests on exactly the same patients as B1-C1.
    fold_col = CONFIG["fold_column"]
    if CONFIG["use_precomputed_folds"] and fold_col in df.columns:
        values = pd.to_numeric(df[fold_col], errors="coerce")
        if values.notna().all():
            ids = sorted(int(v) for v in values.unique())
            if len(ids) == CONFIG["n_splits"]:
                out = []
                for fid in ids:
                    test_idx = np.where(values.values == fid)[0]
                    trainval_idx = np.where(values.values != fid)[0]
                    out.append((fid, trainval_idx, test_idx))

                per_patient = df.groupby("patient_id")[fold_col].nunique()
                assert (per_patient == 1).all(), "a patient appears in more than one fold"
                return out, "precomputed_fold_column(from_preprocessing)"
        print("WARNING: fold column present but unusable, recomputing locally.")
    local, method = make_outer_splits(df)
    print("WARNING: LOCAL fold recompute. These folds are NOT shared with the other configs, "
          "so paired Wilcoxon tests would be invalid.")
    return [(i + 1, tv, te) for i, (tv, te) in enumerate(local)], method + "(LOCAL_RECOMPUTE)"

outer_splits, outer_method = build_outer_splits(metadata)

fold_frames = {}
manifest_rows = []

for fold, trainval_idx, test_idx in outer_splits:
    trainval_df = metadata.iloc[trainval_idx].copy()
    test_df = metadata.iloc[test_idx].copy()

    train_df, val_df, validation_method = make_train_val_split(trainval_df, fold)

    assert set(train_df.patient_id).isdisjoint(set(val_df.patient_id))
    assert set(train_df.patient_id).isdisjoint(set(test_df.patient_id))
    assert set(val_df.patient_id).isdisjoint(set(test_df.patient_id))

    fold_frames[fold] = {
        "train": train_df.reset_index(drop=True),
        "val": val_df.reset_index(drop=True),
        "test": test_df.reset_index(drop=True),
    }

    for split_name, frame in fold_frames[fold].items():
        saved = frame.copy()
        saved["fold"] = fold
        saved["split"] = split_name
        saved.to_csv(SPLIT_ROOT / f"fold_{fold}_{split_name}.csv", index=False)

        manifest_rows.append({
            "fold": fold,
            "split": split_name,
            "outer_method": outer_method,
            "validation_method": validation_method if split_name != "test" else "",
            "patches": len(frame),
            "patients": frame.patient_id.nunique(),
            "benign": int((frame.label == 0).sum()),
            "indeterminate": int((frame.label == 1).sum()),
            "malignant": int((frame.label == 2).sum()),
            "mean_V": float(frame["variance_target"].mean()),
        })

FOLD_IDS = sorted(fold_frames)
print("Fold ids:", FOLD_IDS)

split_manifest = pd.DataFrame(manifest_rows)
split_manifest.to_csv(SPLIT_ROOT / "split_manifest.csv", index=False)

with open(SPLIT_ROOT / "split_config.json", "w") as f:
    json.dump({
        "outer_method": outer_method,
        "n_splits": CONFIG["n_splits"],
        "fold_ids": FOLD_IDS,
        "validation_fraction_of_training_patients": CONFIG["validation_fraction_of_training_patients"],
        "seed": CONFIG["seed"],
        "patients": int(metadata.patient_id.nunique()),
        "patches": int(len(metadata)),
    }, f, indent=2)

# warn early if the indeterminate class is very small
share = metadata["label"].value_counts(normalize=True).reindex([0, 1, 2]).fillna(0)
print("Class share (benign / indeterminate / malignant):", [round(float(x), 3) for x in share])
if share[1] < 0.05:
    print("WARNING: indeterminate < 5%. Plan Sec. 13 says to consider merging to K=2.")

display(split_manifest)


Checking patches:   0%|          | 0/96 [00:00<?, ?it/s]

Fold ids: [1, 2, 3, 4, 5]
Class share (benign / indeterminate / malignant): [0.208, 0.333, 0.458]


,fold,split,outer_method,validation_method,patches,patients,benign,indeterminate,malignant,mean_V
0,1,train,precomputed_fold_column(from_preprocessing),StratifiedShuffleSplit(patient),68,28,17,18,33,0.0
1,1,val,precomputed_fold_column(from_preprocessing),StratifiedShuffleSplit(patient),8,3,1,3,4,0.0
2,1,test,precomputed_fold_column(from_preprocessing),,20,7,2,11,7,0.0
3,2,train,precomputed_fold_column(from_preprocessing),StratifiedShuffleSplit(patient),70,29,18,20,32,0.0
4,2,val,precomputed_fold_column(from_preprocessing),StratifiedShuffleSplit(patient),17,3,2,8,7,0.0
5,2,test,precomputed_fold_column(from_preprocessing),,9,6,0,4,5,0.0
6,3,train,precomputed_fold_column(from_preprocessing),StratifiedShuffleSplit(patient),66,28,10,23,33,0.0
7,3,val,precomputed_fold_column(from_preprocessing),StratifiedShuffleSplit(patient),10,3,1,3,6,0.0
8,3,test,precomputed_fold_column(from_preprocessing),,20,7,9,6,5,0.0
9,4,train,precomputed_fold_column(from_preprocessing),StratifiedShuffleSplit(patient),75,26,17,26,32,0.0


7. Transforms and dataset



In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

AUG = CONFIG["augmentation"]
_shift = AUG["translation_fraction"]
_bilinear = transforms.InterpolationMode.BILINEAR

# Augmentation works on the float tensor (1,H,W) in [0,1], before channel replication and normalisation.
# Empty corners are filled with 0, which is air after the lung window (same as the preprocessing padding).
# flips, +/-15 degree rotation, +/-10% translation.
train_augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=AUG["horizontal_flip_p"]),
    transforms.RandomVerticalFlip(p=AUG["vertical_flip_p"]),
    transforms.RandomRotation(AUG["rotation_degrees"], interpolation=_bilinear, fill=0.0),
    transforms.RandomAffine(degrees=0, translate=(_shift, _shift), interpolation=_bilinear, fill=0.0),
])
normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

# Markers so calls like LIDCC2Dataset(df, eval_transform) keep working
train_transform = "train"
eval_transform = "eval"

def load_patch_array(path):
    # Preprocessing writes float32 .npy patches already scaled to [0,1]. Image files are kept as a fallback.
    path = str(path)
    if path.lower().endswith(".npy"):
        arr = np.load(path).astype(np.float32)
    else:
        with Image.open(path) as image:
            arr = np.asarray(image.convert("L"), dtype=np.float32) / 255.0
    arr = np.squeeze(arr)
    if arr.ndim != 2:
        raise ValueError(f"expected a 2D patch, got shape {arr.shape} for {path}")
    return np.clip(arr, 0.0, 1.0)

def array_to_model_input(arr, training=False):
    x = torch.from_numpy(np.ascontiguousarray(arr, dtype=np.float32)).unsqueeze(0)  # 1,H,W
    size = CONFIG["image_size"]
    if tuple(x.shape[-2:]) != (size, size):
        x = F.interpolate(x.unsqueeze(0), size=(size, size), mode="bilinear", align_corners=False)[0]
    if training:
        x = train_augment(x)
    x = x.repeat(3, 1, 1)          # grayscale -> 3 channels
    return normalize(x)

class LIDCC2Dataset(Dataset):
    def __init__(self, frame, transform):
        self.df = frame.reset_index(drop=True).copy()
        self.training = (transform == train_transform)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = array_to_model_input(load_patch_array(row["image_path"]), training=self.training)

        result = {
            "image": x,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "variance_target": torch.tensor(float(row["variance_target"]), dtype=torch.float32),
            "alignment_mask": torch.tensor(float(row["alignment_mask"]), dtype=torch.float32),
            "reader_distribution": torch.tensor(
                [float(row["q_benign"]), float(row["q_indeterminate"]), float(row["q_malignant"])],
                dtype=torch.float32,
            ),
            "image_path": str(row["image_path"]),
            "patient_id": str(row["patient_id"]),
        }
        if "mask_path" in self.df.columns:
            result["mask_path"] = str(row["mask_path"]) if pd.notna(row["mask_path"]) else ""
        return result

def make_loader(frame, shuffle=False, training=False, batch_size=None):
    generator = torch.Generator()
    generator.manual_seed(CONFIG["seed"])
    return DataLoader(
        LIDCC2Dataset(frame, train_transform if training else eval_transform),
        batch_size=batch_size or CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=generator,
    )


8. MedMamba-T model and optional author-released weights



In [ ]:
def clean_medmamba_state_dict(raw_state):
    state = raw_state

    if isinstance(state, dict):
        for key in ["state_dict", "model", "model_state_dict", "net"]:
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break

    cleaned = {}

    for key, value in state.items():
        clean_key = key

        for prefix in ["module.", "backbone.", "model."]:
            if clean_key.startswith(prefix):
                clean_key = clean_key[len(prefix):]

        if clean_key.startswith("head."):
            continue

        cleaned[clean_key] = value

    return cleaned

class C2MedMambaHAAL(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = VSSM(
            patch_size=4,
            in_chans=3,
            num_classes=0,
            depths=CONFIG["medmamba_depths"],
            dims=CONFIG["medmamba_dims"],
            d_state=CONFIG["d_state"],
            drop_path_rate=CONFIG["drop_path_rate"],
            use_checkpoint=CONFIG["use_checkpoint"],
        )

        self.classifier = nn.Linear(
            int(self.backbone.num_features),
            CONFIG["num_classes"],
        )

        self.pretraining_status = "scratch"

        if MEDMAMBA_WEIGHTS_PATH.exists():
            raw_state = torch.load(
                MEDMAMBA_WEIGHTS_PATH,
                map_location="cpu",
            )

            state = clean_medmamba_state_dict(raw_state)

            incompatible = self.backbone.load_state_dict(
                state,
                strict=False,
            )

            n_model = len(self.backbone.state_dict())
            n_loaded = n_model - len(incompatible.missing_keys)
            if n_loaded / max(n_model, 1) < 0.90:
                # a wrong-size or wrongly named checkpoint would otherwise load almost nothing, silently
                raise RuntimeError(
                    f"Only {n_loaded}/{n_model} backbone tensors matched the weight file. "
                    "Check that it is MedMamba-T and remove it to train from scratch."
                )
            self.pretraining_status = "author_weights_loaded_from_drive"

            print(
                "Loaded author weights.",
                "Missing:", len(incompatible.missing_keys),
                "Unexpected:", len(incompatible.unexpected_keys),
            )
        else:
            print(
                "No MedMamba-T author weights found. "
                "C2 will train from scratch and record this."
            )

    def forward_features(self, x):
        spatial = self.backbone.forward_backbone(x)  # B,H,W,C

        pooled = spatial.permute(0,3,1,2)
        pooled = self.backbone.avgpool(pooled)
        pooled = torch.flatten(pooled, start_dim=1)

        return spatial, pooled

    def forward(self, x):
        _, pooled = self.forward_features(x)
        return self.classifier(pooled)

def build_model():
    return C2MedMambaHAAL().to(DEVICE)

preview_model = build_model()
print(preview_model)
print("Pretraining:", preview_model.pretraining_status)

del preview_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()



No MedMamba-T author weights found. C2 will train from scratch and record this.
C2MedMambaHAAL(
  (backbone): VSSM(
    (patch_embed): PatchEmbed2D(
      (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0): VSSLayer(
        (blocks): ModuleList(
          (0): SS_Conv_SSM(
            (ln_1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (self_attention): SS2D(
              (in_proj): Linear(in_features=48, out_features=192, bias=False)
              (conv2d): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=96)
              (act): SiLU()
              (out_norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
              (out_proj): Linear(in_features=96, out_features=48, bias=False)
            )
            (drop_path): timm.DropPath(0.0)
            (conv33conv33conv11):

9. EDL and HAAL loss



In [ ]:
EPS = CONFIG["epsilon"]

def logits_to_dirichlet(logits, prior_C=None):
    C = CONFIG["prior_c"] if prior_C is None else float(prior_C)
    evidence = F.softplus(logits)
    alpha = evidence + C
    strength = alpha.sum(dim=1, keepdim=True)
    probs = alpha / strength
    vacuity = CONFIG["num_classes"] * C / strength.squeeze(1)
    entropy = -(probs * torch.log(probs + EPS)).sum(dim=1)
    return {
        "evidence": evidence,
        "alpha": alpha,
        "strength": strength.squeeze(1),
        "probs": probs,
        "vacuity": vacuity,
        "aleatoric_entropy": entropy,
        "normalized_aleatoric_entropy": entropy / math.log(CONFIG["num_classes"]),
    }

def expected_dirichlet_ce_soft(alpha, target_distribution):
    S = alpha.sum(dim=1, keepdim=True)
    return (
        target_distribution
        * (torch.digamma(S) - torch.digamma(alpha))
    ).sum(dim=1).mean()

def kl_q_to_p(q, p, eps=1e-8):
    return (q * (torch.log(q + eps) - torch.log(p + eps))).sum(dim=1).mean()

def dirichlet_kl(alpha, beta):
    Sa = alpha.sum(dim=1, keepdim=True)
    Sb = beta.sum(dim=1, keepdim=True)
    log_B_a = torch.lgamma(alpha).sum(dim=1) - torch.lgamma(Sa).squeeze(1)
    log_B_b = torch.lgamma(beta).sum(dim=1) - torch.lgamma(Sb).squeeze(1)
    return (
        log_B_b - log_B_a
        + ((alpha-beta) * (torch.digamma(alpha)-torch.digamma(Sa))).sum(dim=1)
    )

def soft_target_c_consistent_kl(alpha, target, C):
    C = float(C)
    alpha_tilde = target * C + (1.0 - target) * alpha
    beta = torch.full_like(alpha_tilde, C)
    return dirichlet_kl(alpha_tilde, beta).mean()

def dirichlet_kl_to_uniform(alpha_tilde):
    return dirichlet_kl(alpha_tilde, torch.ones_like(alpha_tilde)).mean()

def alignment_weight(epoch, lambda_max, T_warmup):
    if int(T_warmup) == 0:
        return float(lambda_max)
    return float(lambda_max * min(1.0, epoch / float(T_warmup)))

def c2_loss(
    logits,
    targets,
    variance_target,
    epoch,
    prior_C=None,
    lambda_max=None,
    T_warmup=None,
    loss_mode="reader_distribution",
    alignment_mask=None,
    reader_distribution=None,
):
    C = CONFIG["prior_c"] if prior_C is None else float(prior_C)
    lm = CONFIG["lambda_max"] if lambda_max is None else float(lambda_max)
    tw = CONFIG["t_warmup"] if T_warmup is None else int(T_warmup)

    d = logits_to_dirichlet(logits, C)
    y_onehot = F.one_hot(targets, num_classes=CONFIG["num_classes"]).float()

    if loss_mode == "reader_distribution":
        if reader_distribution is None:
            raise ValueError("reader_distribution q is required for loss_mode='reader_distribution'.")
        q = reader_distribution.float()
        if not torch.allclose(q.sum(dim=1), torch.ones_like(q[:, 0]), atol=1e-5):
            raise ValueError("reader_distribution rows must sum to 1.")

        loss_ece = expected_dirichlet_ce_soft(d["alpha"], q)
        loss_align = kl_q_to_p(q, d["probs"], EPS)
        lambda_t = alignment_weight(epoch, lm, tw)

        reg_max = float(CONFIG["evidence_reg_max"])
        reg_warm = int(CONFIG["evidence_reg_warmup"])
        evidence_reg_t = reg_max if reg_warm == 0 else reg_max * min(1.0, epoch / float(reg_warm))
        loss_kl = soft_target_c_consistent_kl(d["alpha"], q, C)
        total = loss_ece + lambda_t * loss_align + evidence_reg_t * loss_kl

    elif loss_mode in ("legacy_v_haal", "haal"):
        loss_ece = expected_dirichlet_ce_soft(d["alpha"], y_onehot)
        squared_error = (d["normalized_aleatoric_entropy"] - variance_target) ** 2
        if alignment_mask is None:
            loss_align = squared_error.mean()
        else:
            valid = alignment_mask.float()
            if valid.sum().item() <= 0:
                raise ValueError(
                    "legacy_v_haal requested but no valid variance-target rows are available."
                )
            loss_align = (squared_error * valid).sum() / valid.sum()
        lambda_t = alignment_weight(epoch, lm, tw)
        loss_kl = torch.zeros((), device=logits.device)
        evidence_reg_t = 0.0
        total = loss_ece + lambda_t * loss_align

    elif loss_mode == "ece_only":
        loss_ece = expected_dirichlet_ce_soft(d["alpha"], y_onehot)
        loss_align = torch.zeros((), device=logits.device)
        loss_kl = torch.zeros((), device=logits.device)
        lambda_t = 0.0
        evidence_reg_t = 0.0
        total = loss_ece

    elif loss_mode == "ece_plus_kl":
        loss_ece = expected_dirichlet_ce_soft(d["alpha"], y_onehot)
        alpha_tilde = y_onehot + (1.0-y_onehot) * d["alpha"]
        loss_kl = dirichlet_kl_to_uniform(alpha_tilde)
        loss_align = torch.zeros((), device=logits.device)
        lambda_t = 0.0
        evidence_reg_t = float(CONFIG["lambda_kl_for_ablation"])
        total = loss_ece + evidence_reg_t * loss_kl

    else:
        raise ValueError(
            f"Unknown loss mode: {loss_mode}. "
            "Supported: reader_distribution, legacy_v_haal, ece_only, ece_plus_kl"
        )

    return total, {
        "loss_total": float(total.detach().item()),
        "loss_ece": float(loss_ece.detach().item()),
        "loss_align": float(loss_align.detach().item()),
        "loss_kl": float(loss_kl.detach().item()),
        "lambda_t": float(lambda_t),
        "evidence_reg_t": float(evidence_reg_t),
    }


In [ ]:
# Pre-training objective audit
_audit_frame = fold_frames[FOLD_IDS[0]]["train"]
_audit_loader = make_loader(_audit_frame, shuffle=False, training=False, batch_size=min(8, CONFIG["batch_size"]))
_audit_batch = next(iter(_audit_loader))

_audit_q = _audit_batch["reader_distribution"].to(DEVICE)
assert torch.isfinite(_audit_q).all()
assert torch.allclose(_audit_q.sum(dim=1), torch.ones(_audit_q.shape[0], device=DEVICE), atol=1e-5)

_audit_model = build_model()
_audit_model.eval()
with torch.no_grad():
    _audit_logits = _audit_model(_audit_batch["image"].to(DEVICE))
    _audit_loss, _audit_parts = c2_loss(
        _audit_logits,
        _audit_batch["label"].to(DEVICE),
        _audit_batch["variance_target"].to(DEVICE),
        epoch=1,
        loss_mode="reader_distribution",
        alignment_mask=_audit_batch["alignment_mask"].to(DEVICE),
        reader_distribution=_audit_q,
    )

print("OBJECTIVE AUDIT PASSED")
print("q row sums:", _audit_q.sum(dim=1).detach().cpu().numpy())
print("q batch mean:", _audit_q.mean(dim=0).detach().cpu().numpy())
print("loss decomposition:", _audit_parts)
if _audit_parts["loss_align"] <= 1e-10:
    raise RuntimeError("Reader-distribution KL alignment is unexpectedly zero at initialization.")

del _audit_model, _audit_batch, _audit_q, _audit_logits
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


No MedMamba-T author weights found. C2 will train from scratch and record this.
OBJECTIVE AUDIT PASSED
q row sums: [1. 1. 1. 1. 1. 1. 1. 1.]
q batch mean: [0.25 0.5  0.25]
loss decomposition: {'loss_total': 6.720097541809082, 'loss_ece': 1.3065564632415771, 'loss_align': 1.0826737880706787, 'loss_kl': 0.17149119079113007, 'lambda_t': 5.0, 'evidence_reg_t': 0.001}


10. Classification, calibration, and alignment metrics



In [ ]:
def multiclass_brier(y_true, probs):
    y = np.eye(3)[np.asarray(y_true, dtype=int)]

    return float(
        np.mean(
            np.sum(
                (np.asarray(probs) - y) ** 2,
                axis=1,
            )
        )
    )

def multiclass_nll(y_true, probs):
    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(probs, dtype=float), 1e-12, 1.0)

    return float(
        -np.mean(
            np.log(
                p[np.arange(len(y)), y]
            )
        )
    )

def ece_score(y_true, probs, bins=15):
    y = np.asarray(y_true)
    p = np.asarray(probs)

    conf = p.max(1)
    pred = p.argmax(1)
    correct = (pred == y).astype(float)

    edges = np.linspace(0,1,bins+1)
    value = 0.0

    for i in range(bins):
        lo, hi = edges[i], edges[i+1]

        mask = (
            (conf >= lo) & (conf <= hi)
            if i == 0
            else (conf > lo) & (conf <= hi)
        )

        if mask.any():
            value += (
                mask.mean()
                * abs(
                    correct[mask].mean()
                    - conf[mask].mean()
                )
            )

    return float(value)

def macro_ovr_auc(y_true, probs):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probs)

    y_bin = label_binarize(
        y,
        classes=[0,1,2],
    )

    values = []

    for k in range(3):
        if len(np.unique(y_bin[:,k])) < 2:
            values.append(np.nan)
        else:
            values.append(
                roc_auc_score(
                    y_bin[:,k],
                    p[:,k],
                )
            )

    finite = [x for x in values if np.isfinite(x)]

    return (
        float(np.mean(finite))
        if finite
        else np.nan
    )

def calculate_metrics(y_true, probs):
    pred = np.asarray(probs).argmax(1)

    return {
        "accuracy": float(
            accuracy_score(y_true, pred)
        ),
        "f1_macro": float(
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "auroc_macro_ovr": macro_ovr_auc(
            y_true,
            probs,
        ),
        "brier_score": multiclass_brier(
            y_true,
            probs,
        ),
        "ece_15": ece_score(
            y_true,
            probs,
            CONFIG["ece_bins"],
        ),
        "nll": multiclass_nll(
            y_true,
            probs,
        ),
    }

def alignment_diagnostics(V, H_normalized):
    V = np.asarray(V, dtype=float)
    H_normalized = np.asarray(H_normalized, dtype=float)

    mse = float(
        np.mean(
            (H_normalized - V) ** 2
        )
    )

    if (
        len(V) >= 2
        and np.std(V) > 0
        and np.std(H_normalized) > 0
    ):
        correlation = float(
            pearsonr(
                V,
                H_normalized,
            ).statistic
        )
    else:
        correlation = np.nan

    return {
        "alignment_mse": mse,
        "entropy_variance_pearson_r": correlation,
    }



11. Single-pass C2 inference



In [ ]:
@torch.no_grad()
def c2_predict(model, loader, prior_C=None):
    model.eval()

    result = {
        "y_true": [],
        "variance_target": [],
        "probs": [],
        "vacuity": [],
        "aleatoric_entropy": [],
        "normalized_aleatoric_entropy": [],
        "strength": [],
        "evidence": [],
    }

    paths = []
    patients = []

    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)

        d = logits_to_dirichlet(
            model(x),
            prior_C,
        )

        result["y_true"].append(
            batch["label"].numpy()
        )

        result["variance_target"].append(
            batch["variance_target"].numpy()
        )

        for key in [
            "probs",
            "vacuity",
            "aleatoric_entropy",
            "normalized_aleatoric_entropy",
            "strength",
            "evidence",
        ]:
            result[key].append(
                d[key].cpu().numpy()
            )

        paths.extend(batch["image_path"])
        patients.extend(batch["patient_id"])

    for key in result:
        result[key] = np.concatenate(result[key])

    result["paths"] = paths
    result["patients"] = patients

    return result



12. Training: AdamW + cosine annealing + validation-Brier early stopping



In [ ]:
def train_one_fold(
    fold,
    train_df,
    val_df,
    prior_C=None,
    lambda_max=None,
    T_warmup=None,
    loss_mode="reader_distribution",
    run_tag="main",
):
    seed_everything(CONFIG["seed"])  # one fixed seed (42)

    model = build_model()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        betas=(
            CONFIG["adam_beta1"],
            CONFIG["adam_beta2"],
        ),
        weight_decay=CONFIG["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG["max_epochs"],
        eta_min=CONFIG["eta_min"],
    )

    train_loader = make_loader(
        train_df,
        shuffle=True,
        training=True,
    )

    val_loader = make_loader(
        val_df,
        shuffle=False,
        training=False,
    )

    C = (
        CONFIG["prior_weight_C"]
        if prior_C is None
        else float(prior_C)
    )

    lm = (
        CONFIG["lambda_max"]
        if lambda_max is None
        else float(lambda_max)
    )

    tw = (
        CONFIG["T_warmup"]
        if T_warmup is None
        else int(T_warmup)
    )

    best_brier = float("inf")
    stale = 0
    history = []

    checkpoint = (
        CHECKPOINT_ROOT
        / f"C2_{run_tag}_fold_{fold}_best.pt"
    )

    for epoch in range(
        1,
        CONFIG["max_epochs"] + 1,
    ):
        model.train()

        sums = {
            "loss_total": 0.0,
            "loss_ece": 0.0,
            "loss_align": 0.0,
            "loss_kl": 0.0,
        }

        seen = 0

        for batch in tqdm(
            train_loader,
            desc=f"C2 {run_tag} fold {fold} epoch {epoch}",
            leave=False,
        ):
            x = batch["image"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)
            V = batch["variance_target"].to(DEVICE, non_blocking=True)
            M = batch["alignment_mask"].to(DEVICE, non_blocking=True)
            Q = batch["reader_distribution"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            loss, parts = c2_loss(
                model(x),
                y,
                V,
                epoch=epoch,
                prior_C=C,
                lambda_max=lm,
                T_warmup=tw,
                loss_mode=loss_mode,
                alignment_mask=M,
                reader_distribution=Q,
            )

            loss.backward()
            optimizer.step()

            batch_size = len(y)
            seen += batch_size

            for key in sums:
                sums[key] += parts[key] * batch_size

        validation = c2_predict(
            model,
            val_loader,
            C,
        )

        val_metrics = calculate_metrics(
            validation["y_true"],
            validation["probs"],
        )

        align = alignment_diagnostics(
            validation["variance_target"],
            validation["normalized_aleatoric_entropy"],
        )

        row = {
            "fold": fold,
            "epoch": epoch,
            "run_tag": run_tag,
            "loss_mode": loss_mode,
            "prior_C": C,
            "lambda_max": lm,
            "T_warmup": tw,
            "lambda_t": (
                alignment_weight(epoch, lm, tw)
                if loss_mode in ("reader_distribution", "legacy_v_haal", "haal")
                else 0.0
            ),
            "lr": optimizer.param_groups[0]["lr"],
            **{
                key: value / max(seen, 1)
                for key, value in sums.items()
            },
            **{
                f"val_{key}": value
                for key, value in val_metrics.items()
            },
            "val_alignment_mse": align["alignment_mse"],
            "val_entropy_variance_pearson_r": (
                align["entropy_variance_pearson_r"]
            ),
        }

        history.append(row)

        print(
            f"fold={fold} epoch={epoch:02d} "
            f"L={row['loss_total']:.4f} "
            f"LECE={row['loss_ece']:.4f} "
            f"LAlign={row['loss_align']:.4f} "
            f"lambda={row['lambda_t']:.3f} "
            f"val_Brier={val_metrics['brier_score']:.4f} "
            f"val_F1={val_metrics['f1_macro']:.4f}"
        )

        if val_metrics["brier_score"] < best_brier - 1e-8:
            best_brier = val_metrics["brier_score"]
            stale = 0

            torch.save({
                "config": CONFIG,
                "fold": fold,
                "epoch": epoch,
                "run_tag": run_tag,
                "loss_mode": loss_mode,
                "prior_C": C,
                "lambda_max": lm,
                "T_warmup": tw,
                "best_val_brier": best_brier,
                "pretraining_status": model.pretraining_status,
                "model_state_dict": model.state_dict(),
            }, checkpoint)

        else:
            stale += 1

        scheduler.step()

        if stale >= CONFIG["early_stopping_patience"]:
            print("Early stopping:", fold, epoch)
            break

    # keep the last-epoch weights too (only the best-Brier checkpoint was saved before)
    torch.save({
        "config": CONFIG, "fold": fold, "epoch": epoch, "run_tag": run_tag,
        "pretraining_status": model.pretraining_status,
        "model_state_dict": model.state_dict(),
    }, CHECKPOINT_ROOT / f"C2_{run_tag}_fold_{fold}_final.pt")

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        LOG_ROOT
        / f"C2_{run_tag}_fold_{fold}_history.csv",
        index=False,
    )

    state = torch.load(
        checkpoint,
        map_location=DEVICE,
    )

    model.load_state_dict(
        state["model_state_dict"]
    )

    model.eval()

    return model, history_df, checkpoint



13. Reliability, rejection, and efficiency



In [ ]:
_trapz = getattr(np, 'trapezoid', None) or np.trapz

def reliability_table(y_true, probs, bins=15):
    y = np.asarray(y_true)
    p = np.asarray(probs)

    conf = p.max(1)
    pred = p.argmax(1)
    correct = pred == y

    edges = np.linspace(0,1,bins+1)
    rows = []

    for i in range(bins):
        lo, hi = edges[i], edges[i+1]

        mask = (
            (conf >= lo) & (conf <= hi)
            if i == 0
            else (conf > lo) & (conf <= hi)
        )

        rows.append({
            "bin": i,
            "lower": lo,
            "upper": hi,
            "count": int(mask.sum()),
            "mean_confidence": (
                float(conf[mask].mean())
                if mask.any()
                else np.nan
            ),
            "accuracy": (
                float(correct[mask].mean())
                if mask.any()
                else np.nan
            ),
        })

    return pd.DataFrame(rows)

def save_reliability(y, p, fold, tag):
    table = reliability_table(
        y,
        p,
        CONFIG["ece_bins"],
    )

    table.to_csv(
        METRICS_ROOT
        / f"C2_{tag}_fold_{fold}_reliability.csv",
        index=False,
    )

def rejection_curve(y, p, uncertainty):
    y = np.asarray(y)
    pred = np.asarray(p).argmax(1)
    order = np.argsort(np.asarray(uncertainty))
    n = len(y)

    return pd.DataFrame([
        {
            "coverage": retained / n,
            "accuracy": float(
                accuracy_score(
                    y[order[:retained]],
                    pred[order[:retained]],
                )
            ),
        }
        for retained in range(1,n+1)
    ])

def random_rejection_curve(y, p, repeats=50, seed=42):
    y = np.asarray(y)
    pred = np.asarray(p).argmax(1)
    n = len(y)

    rng = np.random.RandomState(seed)
    curves = []

    for _ in range(repeats):
        order = rng.permutation(n)

        curves.append([
            accuracy_score(
                y[order[:retained]],
                pred[order[:retained]],
            )
            for retained in range(1,n+1)
        ])

    values = np.asarray(curves)

    return pd.DataFrame({
        "coverage": np.arange(1,n+1) / n,
        "accuracy": values.mean(0),
        "accuracy_sd": values.std(0),
    })

def curve_auc(frame):
    frame = frame.sort_values("coverage")

    return float(
        _trapz(
            frame["accuracy"],
            frame["coverage"],
        )
    )

def save_rejection(y, p, vacuity, fold, tag):
    edl = rejection_curve(y, p, vacuity)

    random_curve = random_rejection_curve(
        y,
        p,
        CONFIG["random_rejection_repeats"],
        CONFIG["seed"] + fold,
    )

    edl.to_csv(
        METRICS_ROOT
        / f"C2_{tag}_fold_{fold}_rejection_vacuity.csv",
        index=False,
    )

    random_curve.to_csv(
        METRICS_ROOT
        / f"C2_{tag}_fold_{fold}_rejection_random.csv",
        index=False,
    )

    return {
        "rejection_auc_vacuity": curve_auc(edl),
        "rejection_auc_random": curve_auc(random_curve),
    }

def ssm_extra_macs(model, input_size=224):
    # thop only counts Conv / Linear / Norm modules. It misses the selective scan and the two einsums
    # inside every SS2D block, so MedMamba would look cheaper than it is
    scan_macs = MEDMAMBA_NS["flops_selective_scan_ref"]   # helper shipped in the official MedMamba.py
    total = 0.0
    resolution = input_size // 4                          # patch embedding has stride 4
    for stage in model.backbone.layers:
        L = resolution * resolution
        for block in stage.blocks:
            s = block.self_attention
            K = 4                                         # four scan directions
            total += scan_macs(B=1, L=L, D=K * s.d_inner, N=s.d_state,
                               with_D=True, with_Z=False, with_Group=True)
            total += K * L * s.d_inner * (s.dt_rank + 2 * s.d_state)   # x_proj einsum
            total += K * L * s.dt_rank * s.d_inner                    # dt_proj einsum
        if stage.downsample is not None:
            resolution //= 2                              # patch merging halves the grid
    return float(total)

def model_efficiency(model, test_loader, warmup_passes=10):
    model.eval()

    total_parameters = sum(p.numel() for p in model.parameters())
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

    dummy = torch.randn(1, 3, CONFIG["image_size"], CONFIG["image_size"], device=DEVICE)

    thop_macs, extra_macs, macs = np.nan, np.nan, np.nan
    try:
        profiled = copy.deepcopy(model)          # thop adds buffers, so profile a copy
        thop_macs = float(profile(profiled, inputs=(dummy,), verbose=False)[0])
        del profiled
        extra_macs = ssm_extra_macs(model, CONFIG["image_size"])
        macs = thop_macs + extra_macs
    except Exception as exc:
        print("MACs profiling warning:", repr(exc))

    # wall-clock per image, averaged over the test set, after warm-up
    with torch.no_grad():
        for _ in range(warmup_passes):
            _ = model(dummy)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    loader = DataLoader(test_loader.dataset, batch_size=1, shuffle=False, num_workers=0)
    times = []
    for batch in loader:
        x = batch["image"].to(DEVICE)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            _ = logits_to_dirichlet(model(x))
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)

    return {
        "total_parameters": int(total_parameters),
        "trainable_parameters": int(trainable_parameters),
        # column name kept for the comparison table; the value is MACs (multiply-accumulate)
        "single_pass_flops": macs,
        "thop_macs_without_scan": thop_macs,
        "ssm_scan_and_einsum_macs": extra_macs,
        "single_pass_latency_ms_per_image": float(np.mean(times)) if times else np.nan,
        "latency_images_timed": len(times),
    }


# **14. MedMamba ESM**

In [ ]:
def final_ss_conv_ssm_block(model):
    return model.backbone.layers[-1].blocks[-1]

def bhwc_heatmap(activations, gradients):
    weights = gradients.mean(
        dim=(1,2),
        keepdim=True,
    )

    heat = F.relu(
        (weights * activations)
        .sum(
            dim=-1,
            keepdim=True,
        )
    )

    heat = heat.permute(0,3,1,2)

    heat = F.interpolate(
        heat,
        size=(224,224),
        mode="bilinear",
        align_corners=False,
    )[0,0].detach().cpu().numpy()

    heat -= heat.min()

    if heat.max() > 0:
        heat /= heat.max()

    return heat

def inverse_cross_scan_average(tensor, H, W):
    # tensor shape: B,4,D,L
    B, K, D, L = tensor.shape

    if K != 4 or L != H*W:
        raise ValueError(
            f"Unexpected internal tensor shape: {tensor.shape}"
        )

    d0 = tensor[:,0].view(B,D,H,W)

    d1 = (
        tensor[:,1]
        .view(B,D,W,H)
        .transpose(2,3)
        .contiguous()
    )

    d2 = (
        torch.flip(
            tensor[:,2],
            dims=[-1],
        )
        .view(B,D,H,W)
    )

    d3 = (
        torch.flip(
            tensor[:,3],
            dims=[-1],
        )
        .view(B,D,W,H)
        .transpose(2,3)
        .contiguous()
    )

    average = (
        d0 + d1 + d2 + d3
    ) / 4.0

    return average.permute(
        0,2,3,1
    ).contiguous()

class MedMambaSaliencyEngine:
    def __init__(self, model, hook_mode="post"):
        self.model = model
        self.hook_mode = hook_mode
        self.block = final_ss_conv_ssm_block(model)
        self.activations = None
        self.gradients = None
        self.handles = []
        self.original_forward_core = None

        if hook_mode == "pre":
            self.handles.append(
                self.block.register_forward_pre_hook(
                    self._pre_hook
                )
            )

        elif hook_mode == "post":
            self.handles.append(
                self.block.register_forward_hook(
                    self._post_hook
                )
            )

        elif hook_mode == "internal":
            self._instrument_internal()

        else:
            raise ValueError(
                f"Unknown hook mode: {hook_mode}"
            )

    def _capture(self, tensor):
        self.activations = tensor
        self.gradients = None

        if tensor.requires_grad:
            tensor.register_hook(
                lambda grad: setattr(
                    self,
                    "gradients",
                    grad,
                )
            )

    def _pre_hook(self, module, inputs):
        self._capture(inputs[0])

    def _post_hook(self, module, inputs, output):
        self._capture(output)

    def _instrument_internal(self):
        ss2d = self.block.self_attention

        self.original_forward_core = ss2d.forward_core

        def instrumented_forward_core(this, x):
            selective_scan_fn = MEDMAMBA_NS.get(
                "selective_scan_fn"
            )

            if selective_scan_fn is None:
                raise RuntimeError(
                    "selective_scan_fn unavailable. "
                    "ESM-Internal cannot run."
                )

            this.selective_scan = selective_scan_fn

            B, C, H, W = x.shape
            L = H * W
            K = 4

            x_hwwh = torch.stack([
                x.view(B, -1, L),
                torch.transpose(
                    x,
                    dim0=2,
                    dim1=3,
                ).contiguous().view(B, -1, L),
            ], dim=1).view(B,2,-1,L)

            xs = torch.cat([
                x_hwwh,
                torch.flip(
                    x_hwwh,
                    dims=[-1],
                ),
            ], dim=1)

            x_dbl = torch.einsum(
                "b k d l, k c d -> b k c l",
                xs.view(B,K,-1,L),
                this.x_proj_weight,
            )

            dts, Bs, Cs = torch.split(
                x_dbl,
                [
                    this.dt_rank,
                    this.d_state,
                    this.d_state,
                ],
                dim=2,
            )

            dts = torch.einsum(
                "b k r l, k d r -> b k d l",
                dts.view(B,K,-1,L),
                this.dt_projs_weight,
            )

            xs_scan = xs.float().view(B,-1,L)
            dts = dts.contiguous().float().view(B,-1,L)
            Bs = Bs.float().view(B,K,-1,L)
            Cs = Cs.float().view(B,K,-1,L)
            Ds = this.Ds.float().view(-1)

            As = -torch.exp(
                this.A_logs.float()
            ).view(-1, this.d_state)

            dt_bias = (
                this.dt_projs_bias
                .float()
                .view(-1)
            )

            out_y = this.selective_scan(
                xs_scan,
                dts,
                As,
                Bs,
                Cs,
                Ds,
                z=None,
                delta_bias=dt_bias,
                delta_softplus=True,
                return_last_state=False,
            ).view(B,K,-1,L)

            this._c2_internal_H = H
            this._c2_internal_W = W

            self._capture(out_y)

            inv_y = torch.flip(
                out_y[:,2:4],
                dims=[-1],
            ).view(B,2,-1,L)

            wh_y = (
                torch.transpose(
                    out_y[:,1].view(
                        B,-1,W,H
                    ),
                    dim0=2,
                    dim1=3,
                )
                .contiguous()
                .view(B,-1,L)
            )

            invwh_y = (
                torch.transpose(
                    inv_y[:,1].view(
                        B,-1,W,H
                    ),
                    dim0=2,
                    dim1=3,
                )
                .contiguous()
                .view(B,-1,L)
            )

            return (
                out_y[:,0],
                inv_y[:,0],
                wh_y,
                invwh_y,
            )

        ss2d.forward_core = types.MethodType(
            instrumented_forward_core,
            ss2d,
        )

    def remove(self):
        for handle in self.handles:
            handle.remove()

        if (
            self.hook_mode == "internal"
            and self.original_forward_core is not None
        ):
            self.block.self_attention.forward_core = (
                self.original_forward_core
            )

    def _heat(self, scalar):
        self.model.zero_grad(set_to_none=True)
        scalar.backward()

        if (
            self.activations is None
            or self.gradients is None
        ):
            raise RuntimeError(
                f"No saliency tensors captured for {self.hook_mode}"
            )

        if self.hook_mode in ["pre", "post"]:
            return bhwc_heatmap(
                self.activations,
                self.gradients,
            )

        H = self.block.self_attention._c2_internal_H
        W = self.block.self_attention._c2_internal_W

        activations_2d = inverse_cross_scan_average(
            self.activations,
            H,
            W,
        )

        gradients_2d = inverse_cross_scan_average(
            self.gradients,
            H,
            W,
        )

        return bhwc_heatmap(
            activations_2d,
            gradients_2d,
        )

    def gradcam(self, x, prior_C=None):
        self.model.eval()

        logits = self.model(x)
        d = logits_to_dirichlet(
            logits,
            prior_C,
        )

        class_index = int(
            d["probs"].argmax(1).item()
        )

        heat = self._heat(
            logits[:,class_index].sum()
        )

        return heat, class_index

    def esm_vacuity(self, x, prior_C=None):
        self.model.eval()

        d = logits_to_dirichlet(
            self.model(x),
            prior_C,
        )

        return self._heat(
            d["vacuity"].sum()
        )

    def esm_aleatoric(self, x, prior_C=None):
        self.model.eval()

        d = logits_to_dirichlet(
            self.model(x),
            prior_C,
        )

        return self._heat(
            d["aleatoric_entropy"].sum()
        )



In [ ]:
# ---- Smoke test: run this before any long training ----
# 1) forward pass and Dirichlet outputs, 2) masked HAAL loss, 3) ESM Pre / Post / Internal each return a 224x224 map
_m = build_model().eval()
_x = torch.randn(2, 3, CONFIG["image_size"], CONFIG["image_size"], device=DEVICE)

with torch.no_grad():
    _logits = _m(_x)
    _spatial, _pooled = _m.forward_features(_x)
assert _logits.shape == (2, 3), _logits.shape
assert torch.isfinite(_logits).all()
print("final spatial tensor (B,H,W,C):", tuple(_spatial.shape))

_d = logits_to_dirichlet(_logits)
assert ((_d["vacuity"] > 0) & (_d["vacuity"] <= 1 + 1e-6)).all()
assert ((_d["normalized_aleatoric_entropy"] >= 0) & (_d["normalized_aleatoric_entropy"] <= 1 + 1e-6)).all()

_y = torch.tensor([0, 2], device=DEVICE)
_v = torch.tensor([0.0, 0.3], device=DEVICE)
_mask = torch.tensor([0.0, 1.0], device=DEVICE)
_q = torch.tensor(
    [[0.75, 0.25, 0.00],
     [0.00, 0.25, 0.75]],
    dtype=torch.float32,
    device=DEVICE,
)
_loss, _parts = c2_loss(
    _logits, _y, _v, epoch=5,
    loss_mode="reader_distribution",
    alignment_mask=_mask,
    reader_distribution=_q,
)
print("loss parts:", _parts)
assert torch.isfinite(_loss)
print("SMOKE LOSS PASSED: reader_distribution q reached c2_loss().")

_probe = torch.randn(1, 3, CONFIG["image_size"], CONFIG["image_size"], device=DEVICE)
for _mode in CONFIG["medmamba_hook_modes"]:
    _engine = MedMambaSaliencyEngine(_m, hook_mode=_mode)
    try:
        _heat = _engine.esm_vacuity(_probe)
        assert _heat.shape == (CONFIG["image_size"], CONFIG["image_size"]) and np.isfinite(_heat).all()
        print(f"ESM-{_mode}: ok")
    finally:
        _engine.remove()

del _m, _x, _probe, _logits, _spatial, _pooled, _d
gc.collect()
torch.cuda.empty_cache()


No MedMamba-T author weights found. C2 will train from scratch and record this.
final spatial tensor (B,H,W,C): (2, 7, 7, 768)
loss parts: {'loss_total': 4.065556526184082, 'loss_ece': 1.3448855876922607, 'loss_align': 0.5440243482589722, 'loss_kl': 0.10985511541366577, 'lambda_t': 5.0, 'evidence_reg_t': 0.005}
SMOKE LOSS PASSED: reader_distribution q reached c2_loss().
ESM-pre: ok
ESM-post: ok
ESM-internal: ok


15. Saliency saving and MedMamba hook-location ablation



In [ ]:
def load_gray01(path):
    arr = load_patch_array(path)
    size = CONFIG["image_size"]
    if arr.shape != (size, size):
        arr = F.interpolate(torch.from_numpy(arr)[None, None], size=(size, size),
                            mode="bilinear", align_corners=False)[0, 0].numpy()
    return arr.astype(np.float32)

def save_saliency_examples(
    model,
    test_df,
    fold,
    hook_mode="post",
    prior_C=None,
    max_examples=None,
):
    max_examples = (
        CONFIG["saliency_max_examples_per_fold"]
        if max_examples is None
        else max_examples
    )

    output_dir = (
        ESM_ROOT
        / f"fold_{fold}"
        / hook_mode
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    dataset = LIDCC2Dataset(
        test_df,
        eval_transform,
    )

    engine = MedMambaSaliencyEngine(
        model,
        hook_mode=hook_mode,
    )

    rows = []

    try:
        for index in tqdm(
            range(
                min(
                    len(dataset),
                    max_examples,
                )
            ),
            desc=f"C2 {hook_mode} saliency fold {fold}",
            leave=False,
        ):
            sample = dataset[index]
            x = sample["image"].unsqueeze(0).to(DEVICE)

            gradcam, predicted = engine.gradcam(
                x,
                prior_C,
            )

            esm_vacuity = engine.esm_vacuity(
                x,
                prior_C,
            )

            esm_aleatoric = engine.esm_aleatoric(
                x,
                prior_C,
            )

            gray = load_gray01(
                sample["image_path"]
            )

            stem = Path(
                sample["image_path"]
            ).stem

            generated = {}

            for name, heat in [
                ("gradcam", gradcam),
                ("esm_vacuity", esm_vacuity),
                ("esm_aleatoric", esm_aleatoric),
            ]:
                npy = output_dir / f"{stem}_{name}.npy"
                png = output_dir / f"{stem}_{name}.png"

                np.save(npy, heat)

                plt.figure(figsize=(5,5))
                plt.imshow(gray, cmap="gray")
                plt.imshow(heat, cmap="jet", alpha=.45)
                plt.axis("off")
                plt.title(f"{hook_mode}: {name}")
                plt.tight_layout()
                plt.savefig(
                    png,
                    dpi=160,
                    bbox_inches="tight",
                )
                plt.close()

                generated[f"{name}_npy"] = str(npy)
                generated[f"{name}_png"] = str(png)

            rows.append({
                "fold": fold,
                "hook_mode": hook_mode,
                "image_path": sample["image_path"],
                "patient_id": sample["patient_id"],
                "variance_target": float(
                    sample["variance_target"].item()
                ),
                "predicted_label": predicted,
                **generated,
            })

    finally:
        engine.remove()

    output = pd.DataFrame(rows)

    output.to_csv(
        output_dir / "saliency_manifest.csv",
        index=False,
    )

    return output

RUN_HOOK_LOCATION_ABLATION = True  #False

def run_hook_location_ablation(
    trained_model,
    test_df,
    fold=0,
):
    rows = []

    for hook_mode in CONFIG["medmamba_hook_modes"]:
        try:
            manifest = save_saliency_examples(
                trained_model,
                test_df,
                fold,
                hook_mode=hook_mode,
                prior_C=CONFIG["prior_weight_C"],
            )

            rows.append({
                "hook_mode": hook_mode,
                "status": "generated",
                "cases": len(manifest),
            })

        except Exception as exc:
            rows.append({
                "hook_mode": hook_mode,
                "status": "failed",
                "cases": 0,
                "error": repr(exc),
            })

    output = pd.DataFrame(rows)

    output.to_csv(
        ABLATION_ROOT
        / f"C2_fold{fold}_ESM_hook_location_status.csv",
        index=False,
    )

    return output



16. Optional contour-grounded XAI metrics



In [ ]:
def load_binary_mask(path):
    # ground truth = union of the contours from ALL annotating radiologists.
    path = str(path)
    if path.lower().endswith(".npy"):
        arr = np.squeeze(np.load(path))
    else:
        with Image.open(path) as image:
            arr = np.asarray(image.convert("L"))
    size = CONFIG["image_size"]
    if arr.shape != (size, size):
        arr = np.asarray(Image.fromarray((arr > 0).astype(np.uint8) * 255).resize((size, size), Image.Resampling.NEAREST))
    return (arr > 0).astype(np.uint8)

def pointing_game(heat, mask):
    y, x = np.unravel_index(
        np.argmax(heat),
        heat.shape,
    )

    return int(mask[y,x] > 0)

def boundary_distance(heat, mask):
    mask_bool = mask.astype(bool)

    if not mask_bool.any():
        return np.nan

    tensor = torch.tensor(
        mask_bool.astype(np.float32)
    )[None,None]

    eroded = -F.max_pool2d(
        -tensor,
        3,
        1,
        1,
    )

    boundary = (
        mask_bool
        & ~(
            eroded[0,0].numpy()
            > 0.5
        )
    )

    ys, xs = np.where(boundary)

    if len(ys) == 0:
        return np.nan

    py, px = np.unravel_index(
        np.argmax(heat),
        heat.shape,
    )

    return float(
        np.sqrt(
            (ys-py)**2
            + (xs-px)**2
        ).min()
    )

def gray01_to_tensor(gray):
    # stays in float, no 8-bit rounding, same path as the training input
    return array_to_model_input(np.clip(gray, 0, 1).astype(np.float32), training=False).unsqueeze(0).to(DEVICE)

@torch.no_grad()
def class_probability(
    model,
    gray,
    class_index,
    prior_C=None,
):
    d = logits_to_dirichlet(
        model(
            gray01_to_tensor(gray)
        ),
        prior_C,
    )

    return float(
        d["probs"][
            0,
            class_index,
        ].item()
    )

def insertion_deletion_auc(
    model,
    gray,
    heat,
    class_index,
    prior_C=None,
    steps=20,
):
    h, w = gray.shape
    n_pixels = h*w

    order = np.argsort(
        heat.reshape(-1)
    )[::-1]

    tensor = torch.tensor(
        gray,
        dtype=torch.float32,
    )[None,None]

    baseline = F.avg_pool2d(
        tensor,
        31,
        1,
        15,
    )[0,0].numpy()

    fractions = np.linspace(
        0,
        1,
        steps+1,
    )

    insertion = []
    deletion = []

    original_flat = gray.reshape(-1)
    baseline_flat = baseline.reshape(-1)

    for fraction in fractions:
        k = int(round(fraction * n_pixels))
        selected = order[:k]

        inserted = baseline_flat.copy()
        inserted[selected] = original_flat[selected]

        deleted = original_flat.copy()
        deleted[selected] = baseline_flat[selected]

        insertion.append(
            class_probability(
                model,
                inserted.reshape(h,w),
                class_index,
                prior_C,
            )
        )

        deletion.append(
            class_probability(
                model,
                deleted.reshape(h,w),
                class_index,
                prior_C,
            )
        )

    return (
        float(_trapz(insertion, fractions)),
        float(_trapz(deletion, fractions)),
    )

def evaluate_xai_faithfulness(
    model,
    test_df,
    fold,
    hook_mode="post",
    prior_C=None,
):
    # Insertion/Deletion needs no contour. Pointing Game and Boundary Distance do.
    has_mask_col = "mask_path" in test_df.columns
    if has_mask_col:
        has_mask = test_df["mask_path"].notna() & test_df["mask_path"].astype(str).map(
            lambda p: p != "" and Path(p).exists()
        )
    else:
        has_mask = pd.Series(False, index=test_df.index)

    frame = test_df.reset_index(drop=True)
    has_mask = has_mask.reset_index(drop=True)
    n_masks = int(has_mask.sum())

    if n_masks == 0:
        status = "insertion_deletion_only_no_contour_masks"
    elif n_masks < len(frame):
        status = f"computed_masks_for_{n_masks}_of_{len(frame)}_nodules"
    else:
        status = "computed"

    dataset = LIDCC2Dataset(frame, eval_transform)
    engine = MedMambaSaliencyEngine(model, hook_mode=hook_mode)
    rows = []

    try:
        for index in tqdm(range(len(dataset)), desc=f"C2 XAI {hook_mode} fold {fold}", leave=False):
            sample = dataset[index]
            row = frame.iloc[index]
            x = sample["image"].unsqueeze(0).to(DEVICE)

            gradcam, predicted = engine.gradcam(x, prior_C)
            esm_vacuity = engine.esm_vacuity(x, prior_C)
            esm_aleatoric = engine.esm_aleatoric(x, prior_C)

            mask = load_binary_mask(row["mask_path"]) if bool(has_mask.iloc[index]) else None
            gray = load_gray01(sample["image_path"])

            for target, heat in [("gradcam", gradcam), ("esm_vacuity", esm_vacuity), ("esm_aleatoric", esm_aleatoric)]:
                insertion_auc, deletion_auc = insertion_deletion_auc(
                    model, gray, heat, predicted, prior_C, CONFIG["xai_metric_steps"]
                )
                rows.append({
                    "image_path": sample["image_path"],
                    "mask_path": row["mask_path"] if mask is not None else "",
                    "hook_mode": hook_mode,
                    "target": target,
                    "pointing_hit": pointing_game(heat, mask) if mask is not None else np.nan,
                    "boundary_distance_px": boundary_distance(heat, mask) if mask is not None else np.nan,
                    "insertion_auc": insertion_auc,
                    "deletion_auc": deletion_auc,
                })
    finally:
        engine.remove()

    result_df = pd.DataFrame(rows)
    result_df.to_csv(METRICS_ROOT / f"C2_fold_{fold}_{hook_mode}_xai_faithfulness.csv", index=False)

    summary = {"xai_quantitative_status": status}
    for target, group in result_df.groupby("target"):
        summary[f"{hook_mode}_{target}_pointing_game_accuracy"] = float(group["pointing_hit"].mean())
        summary[f"{hook_mode}_{target}_boundary_distance_px"] = float(group["boundary_distance_px"].mean())
        summary[f"{hook_mode}_{target}_insertion_auc"] = float(group["insertion_auc"].mean())
        summary[f"{hook_mode}_{target}_deletion_auc"] = float(group["deletion_auc"].mean())
    return summary


17. Prediction saving and fold evaluation



In [ ]:
def save_predictions(result, fold, tag="main"):
    probs = result["probs"]
    y = result["y_true"]
    pred = probs.argmax(1)

    frame = pd.DataFrame({
        "fold": fold,
        "image_path": result["paths"],
        "patient_id": result["patients"],
        "true_label": y,
        "predicted_label": pred,
        "p_benign": probs[:,0],
        "p_indeterminate": probs[:,1],
        "p_malignant": probs[:,2],
        "variance_target": result["variance_target"],
        "vacuity": result["vacuity"],
        "aleatoric_entropy": result["aleatoric_entropy"],
        "normalized_aleatoric_entropy": (
            result["normalized_aleatoric_entropy"]
        ),
        "alignment_residual": (
            result["normalized_aleatoric_entropy"]
            - result["variance_target"]
        ),
        "dirichlet_strength": result["strength"],
        "evidence_benign": result["evidence"][:,0],
        "evidence_indeterminate": result["evidence"][:,1],
        "evidence_malignant": result["evidence"][:,2],
    })

    frame.to_csv(
        PREDICTION_ROOT
        / f"C2_{tag}_fold_{fold}_predictions.csv",
        index=False,
    )

    return frame

def evaluate_fold(
    model,
    test_df,
    fold,
    prior_C=None,
    tag="main",
    primary_hook_mode="post",
    save_xai=True,
    lambda_max=None,
    T_warmup=None,
):
    test_loader = make_loader(
        test_df,
        shuffle=False,
        training=False,
    )

    result = c2_predict(
        model,
        test_loader,
        prior_C,
    )

    save_predictions(
        result,
        fold,
        tag,
    )

    base_metrics = calculate_metrics(
        result["y_true"],
        result["probs"],
    )

    alignment_metrics = alignment_diagnostics(
        result["variance_target"],
        result["normalized_aleatoric_entropy"],
    )

    save_reliability(
        result["y_true"],
        result["probs"],
        fold,
        tag,
    )

    rejection_metrics = save_rejection(
        result["y_true"],
        result["probs"],
        result["vacuity"],
        fold,
        tag,
    )

    efficiency_metrics = model_efficiency(
        model,
        test_loader,
    )

    xai_metrics = {
        "xai_quantitative_status": "not_run"
    }

    if save_xai:
        save_saliency_examples(
            model,
            test_df,
            fold,
            hook_mode=primary_hook_mode,
            prior_C=prior_C,
        )

        xai_metrics = evaluate_xai_faithfulness(
            model,
            test_df,
            fold,
            hook_mode=primary_hook_mode,
            prior_C=prior_C,
        )

    pred = result["probs"].argmax(1)

    np.save(
        METRICS_ROOT
        / f"C2_{tag}_fold_{fold}_confusion_matrix.npy",
        confusion_matrix(
            result["y_true"],
            pred,
            labels=[0,1,2],
        ),
    )

    with open(
        METRICS_ROOT
        / f"C2_{tag}_fold_{fold}_classification_report.json",
        "w",
    ) as f:
        json.dump(
            classification_report(
                result["y_true"],
                pred,
                labels=[0,1,2],
                target_names=CONFIG["class_names"],
                output_dict=True,
                zero_division=0,
            ),
            f,
            indent=2,
        )

    row = {
        "fold": fold,
        "prior_C": (
            CONFIG["prior_weight_C"]
            if prior_C is None
            else float(prior_C)
        ),
        "lambda_max": CONFIG["lambda_max"] if lambda_max is None else float(lambda_max),
        "T_warmup": CONFIG["T_warmup"] if T_warmup is None else int(T_warmup),
        "test_patches": len(test_df),
        "test_patients": test_df.patient_id.nunique(),
        "pretraining_status": model.pretraining_status,
        "primary_esm_hook": primary_hook_mode,
        **base_metrics,
        **rejection_metrics,
        **efficiency_metrics,
        **xai_metrics,
        **alignment_metrics,
        "mean_variance_target": float(
            np.mean(result["variance_target"])
        ),
        "mean_vacuity": float(
            np.mean(result["vacuity"])
        ),
        "mean_aleatoric_entropy": float(
            np.mean(result["aleatoric_entropy"])
        ),
        "mean_dirichlet_strength": float(
            np.mean(result["strength"])
        ),
    }

    pd.DataFrame([row]).to_csv(
        METRICS_ROOT
        / f"C2_{tag}_fold_{fold}_metrics.csv",
        index=False,
    )

    return row



18. Main five-fold C2
Post is the initial robust spatial hook. Run the C2 hook ablation on fold 0,
then freeze the chosen hook for final XAI reporting if required.



In [ ]:
PRIMARY_ESM_HOOK = "post"
RUN_MAIN_5FOLD = True
RUN_HOOK_LOCATION_ABLATION = True   #False

all_fold_metrics = []

if RUN_MAIN_5FOLD:
    for fold in FOLD_IDS:
        print("\n" + "="*80)
        print(f"C2 MAIN FOLD {fold}")
        print("="*80)

        train_df = fold_frames[fold]["train"]
        val_df = fold_frames[fold]["val"]
        test_df = fold_frames[fold]["test"]

        model, history, checkpoint = train_one_fold(
            fold,
            train_df,
            val_df,
            prior_C=CONFIG["prior_weight_C"],
            lambda_max=CONFIG["lambda_max"],
            T_warmup=CONFIG["T_warmup"],
            loss_mode="reader_distribution",
            run_tag="main",
        )

        fold_result = evaluate_fold(
            model,
            test_df,
            fold,
            prior_C=CONFIG["prior_weight_C"],
            tag="main",
            primary_hook_mode=PRIMARY_ESM_HOOK,
            save_xai=True,
        )

        all_fold_metrics.append(fold_result)

        if fold == FOLD_IDS[0] and RUN_HOOK_LOCATION_ABLATION:
            display(
                run_hook_location_ablation(
                    model,
                    test_df,
                    fold=FOLD_IDS[0],
                )
            )

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_metrics_df = pd.DataFrame(all_fold_metrics)

    fold_metrics_df.to_csv(
        METRICS_ROOT / "C2_fold_metrics.csv",
        index=False,
    )

    display(fold_metrics_df)




C2 MAIN FOLD 1
No MedMamba-T author weights found. C2 will train from scratch and record this.


C2 main fold 1 epoch 1:   0%|          | 0/3 [00:00<?, ?it/s]

fold=1 epoch=01 L=6.6306 LECE=1.2740 LAlign=1.0713 lambda=5.000 val_Brier=0.6512 val_F1=0.2222


C2 main fold 1 epoch 2:   0%|          | 0/3 [00:00<?, ?it/s]

fold=1 epoch=02 L=6.6055 LECE=1.2501 LAlign=1.0709 lambda=5.000 val_Brier=0.7047 val_F1=0.2424


...

C2 main fold 5 epoch 12:   0%|          | 0/2 [00:00<?, ?it/s]

fold=5 epoch=12 L=5.9987 LECE=1.0970 LAlign=0.9791 lambda=5.000 val_Brier=0.6414 val_F1=0.2444
Early stopping: 5 12


C2 post saliency fold 5:   0%|          | 0/30 [00:00<?, ?it/s]

C2 XAI post fold 5:   0%|          | 0/33 [00:00<?, ?it/s]

,fold,prior_C,lambda_max,T_warmup,test_patches,test_patients,pretraining_status,primary_esm_hook,accuracy,f1_macro,...,post_gradcam_pointing_game_accuracy,post_gradcam_boundary_distance_px,post_gradcam_insertion_auc,post_gradcam_deletion_auc,alignment_mse,entropy_variance_pearson_r,mean_variance_target,mean_vacuity,mean_aleatoric_entropy,mean_dirichlet_strength
0,1,1.0,5.0,0,20,7,scratch,post,0.350000,0.172840,...,NaN,NaN,0.486621,0.486999,0.851783,NaN,0.0,0.288271,1.013426,10.415929
1,2,1.0,5.0,0,9,6,scratch,post,0.555556,0.550000,...,NaN,NaN,0.391782,0.395126,0.930453,NaN,0.0,0.218409,1.058500,13.739175
2,3,1.0,5.0,0,20,7,scratch,post,0.250000,0.133333,...,NaN,NaN,0.545122,0.548046,0.723952,NaN,0.0,0.176723,0.930466,17.033152
3,4,1.0,5.0,0,14,9,scratch,post,0.571429,0.242424,...,NaN,NaN,0.510163,0.504078,0.773956,NaN,0.0,0.180753,0.957778,16.611395
4,5,1.0,5.0,0,33,9,scratch,post,0.545455,0.235294,...,NaN,NaN,0.456566,0.452851,0.842294,NaN,0.0,0.411485,1.008251,7.291503


19. Five-fold mean ± SD



In [ ]:
if RUN_MAIN_5FOLD and all_fold_metrics:
    df = pd.DataFrame(all_fold_metrics)
    rows = []

    for metric in df.columns:
        if (
            metric == "fold"
            or not pd.api.types.is_numeric_dtype(df[metric])
        ):
            continue

        values = pd.to_numeric(
            df[metric],
            errors="coerce",
        )

        rows.append({
            "metric": metric,
            "mean": (
                float(values.mean())
                if values.notna().any()
                else np.nan
            ),
            "std": (
                float(values.std(ddof=1))
                if values.notna().sum() > 1
                else np.nan
            ),
            "n_valid_folds": int(
                values.notna().sum()
            ),
        })

    summary = pd.DataFrame(rows)

    summary.to_csv(
        METRICS_ROOT / "C2_5fold_mean_std.csv",
        index=False,
    )

    display(summary)



,metric,mean,std,n_valid_folds
0,prior_C,1.000000e+00,0.000000,5
1,lambda_max,5.000000e+00,0.000000,5
2,T_warmup,0.000000e+00,0.000000,5
3,test_patches,1.920000e+01,8.983318,5
4,test_patients,7.600000e+00,1.341641,5
5,accuracy,4.544877e-01,0.145686,5
6,f1_macro,2.667782e-01,0.164644,5
7,auroc_macro_ovr,5.192016e-01,0.115615,5
8,brier_score,6.670050e-01,0.102651,5
9,ece_15,2.064664e-01,0.113932,5


19b. Figures and pooled predictions

In [ ]:
from sklearn.metrics import roc_curve

def load_all_predictions(tag="main"):
    frames = []
    for f in FOLD_IDS:
        p = PREDICTION_ROOT / f"C2_{tag}_fold_{f}_predictions.csv"
        if p.exists():
            frames.append(pd.read_csv(p))
    return pd.concat(frames, ignore_index=True) if frames else None

def make_c2_figures(tag="main"):
    pooled = load_all_predictions(tag)
    if pooled is None:
        print("No predictions found. Run the main five-fold cell first.")
        return
    pooled.to_csv(PREDICTION_ROOT / f"C2_{tag}_all_folds_predictions.csv", index=False)

    y = pooled["true_label"].values
    probs = pooled[["p_benign", "p_indeterminate", "p_malignant"]].values
    pred = probs.argmax(1)
    names = CONFIG["class_names"]

    # training curves, one line per fold
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for f in FOLD_IDS:
        h = LOG_ROOT / f"C2_{tag}_fold_{f}_history.csv"
        if not h.exists():
            continue
        hist = pd.read_csv(h)
        axes[0].plot(hist["epoch"], hist["loss_total"], label=f"fold {f}")
        axes[1].plot(hist["epoch"], hist["val_brier_score"], label=f"fold {f}")
        axes[2].plot(hist["epoch"], hist["loss_align"], label=f"fold {f}")
    axes[0].set_title("Train loss (L_HAAL)")
    axes[1].set_title("Validation Brier score")
    axes[2].set_title("L_Align (should fall after warm-up)")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(PLOT_ROOT / f"C2_{tag}_training_curves.png", dpi=160)
    plt.close(fig)

    # pooled confusion matrix
    cm = confusion_matrix(y, pred, labels=[0, 1, 2])
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels(names, rotation=30)
    ax.set_yticks(range(3)); ax.set_yticklabels(names)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("C2 confusion matrix (all folds)")
    fig.tight_layout()
    fig.savefig(PLOT_ROOT / f"C2_{tag}_confusion_matrix.png", dpi=160)
    plt.close(fig)

    # one-vs-rest ROC
    fig, ax = plt.subplots(figsize=(5, 4.5))
    for k, name in enumerate(names):
        yk = (y == k).astype(int)
        if yk.min() == yk.max():
            continue
        fpr, tpr, _ = roc_curve(yk, probs[:, k])
        ax.plot(fpr, tpr, label=name)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
    ax.set_title("C2 ROC, one-vs-rest (all folds)"); ax.legend()
    fig.tight_layout()
    fig.savefig(PLOT_ROOT / f"C2_{tag}_roc_curves.png", dpi=160)
    plt.close(fig)

    # reliability diagram
    rel = reliability_table(y, probs, CONFIG["ece_bins"]).dropna()
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.plot(rel["mean_confidence"], rel["accuracy"], "o-")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
    ax.set_title(f"C2 reliability, {CONFIG['ece_bins']} bins (all folds)")
    fig.tight_layout()
    fig.savefig(PLOT_ROOT / f"C2_{tag}_reliability.png", dpi=160)
    plt.close(fig)

    # rejection curve: vacuity vs random
    edl = rejection_curve(y, probs, pooled["vacuity"].values)
    rnd = random_rejection_curve(y, probs, CONFIG["random_rejection_repeats"], CONFIG["seed"])
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ax.plot(edl["coverage"], edl["accuracy"], label="reject by vacuity u")
    ax.plot(rnd["coverage"], rnd["accuracy"], "--", label="random rejection")
    ax.set_xlabel("coverage"); ax.set_ylabel("accuracy"); ax.set_title("C2 rejection curve (all folds)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOT_ROOT / f"C2_{tag}_rejection_curve.png", dpi=160)
    plt.close(fig)
    print("Figures saved to", PLOT_ROOT)

if RUN_MAIN_5FOLD and all_fold_metrics:
    make_c2_figures("main")


Figures saved to /content/drive/MyDrive/LIDC_Thesis/experiments/C2_MedMambaT_HAAL/plots


# **20. Quantitative ESM hook-location comparison**

In [ ]:
def compare_hook_location_metrics(
    model,
    test_df,
    fold=0,
):
    rows = []

    for hook_mode in CONFIG["medmamba_hook_modes"]:
        try:
            result = evaluate_xai_faithfulness(
                model,
                test_df,
                fold,
                hook_mode=hook_mode,
                prior_C=CONFIG["prior_weight_C"],
            )

            status = result.pop(
                "xai_quantitative_status",
                "computed",
            )

            rows.append({
                "hook_mode": hook_mode,
                "status": status,
                **result,
            })

        except Exception as exc:
            rows.append({
                "hook_mode": hook_mode,
                "status": "failed",
                "error": repr(exc),
            })

    output = pd.DataFrame(rows)

    output.to_csv(
        ABLATION_ROOT
        / f"C2_fold{fold}_ESM_hook_location_metrics.csv",
        index=False,
    )

    return output



# **20b. Run the hook comparison from the saved checkpoints**

In [ ]:
RUN_HOOK_COMPARISON = True
HOOK_COMPARISON_FOLDS = FOLD_IDS

def load_trained_fold_model(fold, run_tag="main"):
    state = torch.load(CHECKPOINT_ROOT / f"C2_{run_tag}_fold_{fold}_best.pt", map_location=DEVICE)
    model = build_model()
    model.load_state_dict(state["model_state_dict"])
    model.eval()
    return model

hook_tables = []
if RUN_HOOK_COMPARISON:
    for fold in HOOK_COMPARISON_FOLDS:
        if not (CHECKPOINT_ROOT / f"C2_main_fold_{fold}_best.pt").exists():
            print(f"fold {fold}: no checkpoint yet, skipped")
            continue
        model = load_trained_fold_model(fold)
        table = compare_hook_location_metrics(model, fold_frames[fold]["test"], fold=fold)
        table.insert(0, "fold", fold)
        hook_tables.append(table)
        del model
        gc.collect()
        torch.cuda.empty_cache()

    if hook_tables:
        all_hooks = pd.concat(hook_tables, ignore_index=True)
        all_hooks.to_csv(ABLATION_ROOT / "C2_ESM_hook_location_all_folds.csv", index=False)
        display(all_hooks)


No MedMamba-T author weights found. C2 will train from scratch and record this.


C2 XAI pre fold 1:   0%|          | 0/20 [00:00<?, ?it/s]

C2 XAI post fold 1:   0%|          | 0/20 [00:00<?, ?it/s]

C2 XAI internal fold 1:   0%|          | 0/20 [00:00<?, ?it/s]

No MedMamba-T author weights found. C2 will train from scratch and record this.


...

No MedMamba-T author weights found. C2 will train from scratch and record this.


C2 XAI pre fold 5:   0%|          | 0/33 [00:00<?, ?it/s]

C2 XAI post fold 5:   0%|          | 0/33 [00:00<?, ?it/s]

C2 XAI internal fold 5:   0%|          | 0/33 [00:00<?, ?it/s]

,fold,hook_mode,status,pre_esm_aleatoric_pointing_game_accuracy,pre_esm_aleatoric_boundary_distance_px,pre_esm_aleatoric_insertion_auc,pre_esm_aleatoric_deletion_auc,pre_esm_vacuity_pointing_game_accuracy,pre_esm_vacuity_boundary_distance_px,pre_esm_vacuity_insertion_auc,...,internal_esm_aleatoric_insertion_auc,internal_esm_aleatoric_deletion_auc,internal_esm_vacuity_pointing_game_accuracy,internal_esm_vacuity_boundary_distance_px,internal_esm_vacuity_insertion_auc,internal_esm_vacuity_deletion_auc,internal_gradcam_pointing_game_accuracy,internal_gradcam_boundary_distance_px,internal_gradcam_insertion_auc,internal_gradcam_deletion_auc
0,1,pre,insertion_deletion_only_no_contour_masks,NaN,NaN,0.486435,0.486942,NaN,NaN,0.486165,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,post,insertion_deletion_only_no_contour_masks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,internal,insertion_deletion_only_no_contour_masks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.486590,0.486807,NaN,NaN,0.487021,0.486602,NaN,NaN,0.486923,0.486618
3,2,pre,insertion_deletion_only_no_contour_masks,NaN,NaN,0.394694,0.392319,NaN,NaN,0.393969,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,post,insertion_deletion_only_no_contour_masks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2,internal,insertion_deletion_only_no_contour_masks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.393012,0.394427,NaN,NaN,0.393015,0.394270,NaN,NaN,0.394491,0.392771
6,3,pre,insertion_deletion_only_no_contour_masks,NaN,NaN,0.546706,0.546711,NaN,NaN,0.546656,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,3,post,insertion_deletion_only_no_contour_masks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,3,internal,insertion_deletion_only_no_contour_masks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.546654,0.547663,NaN,NaN,0.549050,0.544468,NaN,NaN,0.544510,0.549482
9,4,pre,insertion_deletion_only_no_contour_masks,NaN,NaN,0.500593,0.512802,NaN,NaN,0.507701,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# **21. Optional C2 sensitivity and loss-isolation reruns**

In [ ]:
RUN_C2_LAMBDA_RECHECK = False
RUN_C2_TWARMUP_RECHECK = False
RUN_C2_PRIOR_RECHECK = False
RUN_C2_LOSS_ISOLATION = False

def run_fold0_parameter_ablation(parameter, values):
    train_df = fold_frames[FOLD_IDS[0]]["train"]
    val_df = fold_frames[FOLD_IDS[0]]["val"]
    test_df = fold_frames[FOLD_IDS[0]]["test"]

    rows = []

    for value in values:
        C = CONFIG["prior_weight_C"]
        lm = CONFIG["lambda_max"]
        tw = CONFIG["T_warmup"]

        if parameter == "prior_C":
            C = float(value)
        elif parameter == "lambda_max":
            lm = float(value)
        elif parameter == "T_warmup":
            tw = int(value)
        else:
            raise ValueError(parameter)

        tag = (
            f"sensitivity_{parameter}_"
            + str(value).replace(".", "p")
        )

        model, history, _ = train_one_fold(
            FOLD_IDS[0],
            train_df,
            val_df,
            prior_C=C,
            lambda_max=lm,
            T_warmup=tw,
            loss_mode="reader_distribution",
            run_tag=tag,
        )

        result = evaluate_fold(
            model,
            test_df,
            FOLD_IDS[0],
            prior_C=C,
            lambda_max=lm,
            T_warmup=tw,
            tag=tag,
            primary_hook_mode=PRIMARY_ESM_HOOK,
            save_xai=False,
        )

        best_index = history["val_brier_score"].idxmin()

        result.update({
            "ablation": parameter,
            "ablation_value": value,
            "best_val_brier": float(
                history.loc[
                    best_index,
                    "val_brier_score",
                ]
            ),
            "best_val_f1_macro": float(
                history.loc[
                    best_index,
                    "val_f1_macro",
                ]
            ),
        })

        rows.append(result)

        del model
        gc.collect()

    output = pd.DataFrame(rows)

    output.to_csv(
        ABLATION_ROOT
        / f"C2_fold0_{parameter}_sensitivity.csv",
        index=False,
    )

    return output

def run_fold0_loss_isolation():
    train_df = fold_frames[FOLD_IDS[0]]["train"]
    val_df = fold_frames[FOLD_IDS[0]]["val"]
    test_df = fold_frames[FOLD_IDS[0]]["test"]

    rows = []

    for loss_mode in CONFIG["ablation_loss_modes"]:
        tag = f"loss_{loss_mode}"

        model, history, _ = train_one_fold(
            FOLD_IDS[0],
            train_df,
            val_df,
            prior_C=CONFIG["prior_weight_C"],
            lambda_max=CONFIG["lambda_max"],
            T_warmup=CONFIG["T_warmup"],
            loss_mode=loss_mode,
            run_tag=tag,
        )

        result = evaluate_fold(
            model,
            test_df,
            FOLD_IDS[0],
            prior_C=CONFIG["prior_weight_C"],
            tag=tag,
            primary_hook_mode=PRIMARY_ESM_HOOK,
            save_xai=False,
        )

        best_index = history["val_brier_score"].idxmin()

        result.update({
            "ablation": "loss_component",
            "ablation_value": loss_mode,
            "best_val_brier": float(
                history.loc[
                    best_index,
                    "val_brier_score",
                ]
            ),
        })

        rows.append(result)

        del model
        gc.collect()

    output = pd.DataFrame(rows)

    output.to_csv(
        ABLATION_ROOT
        / "C2_fold0_loss_component_isolation.csv",
        index=False,
    )

    return output

if RUN_C2_LAMBDA_RECHECK:
    display(
        run_fold0_parameter_ablation(
            "lambda_max",
            CONFIG["ablation_lambda_max_values"],
        )
    )

if RUN_C2_TWARMUP_RECHECK:
    display(
        run_fold0_parameter_ablation(
            "T_warmup",
            CONFIG["ablation_T_warmup_values"],
        )
    )

if RUN_C2_PRIOR_RECHECK:
    display(
        run_fold0_parameter_ablation(
            "prior_C",
            CONFIG["ablation_prior_C_values"],
        )
    )

if RUN_C2_LOSS_ISOLATION:
    display(
        run_fold0_loss_isolation()
    )


22. Paired five-fold Wilcoxon helper



In [ ]:
def paired_wilcoxon_comparison(
    c2_csv,
    other_csv,
    metric,
    other_name="C1",
):
    c2 = pd.read_csv(
        c2_csv
    )[["fold", metric]].rename(
        columns={metric: "C2"}
    )

    other = pd.read_csv(
        other_csv
    )[["fold", metric]].rename(
        columns={metric: other_name}
    )

    paired = c2.merge(
        other,
        on="fold",
        how="inner",
    ).dropna()

    if len(paired) != 5:
        raise ValueError(
            f"Formal comparison requires 5 paired folds; found {len(paired)}"
        )

    statistic, p_value = wilcoxon(
        paired["C2"].values,
        paired[other_name].values,
        alternative="two-sided",
        mode="auto",
    )

    difference = (
        paired["C2"].values
        - paired[other_name].values
    )

    return {
        "metric": metric,
        "n_pairs": 5,
        "wilcoxon_statistic": float(statistic),
        "p_value": float(p_value),
        "alpha": 0.05,
        "C2_mean": float(paired["C2"].mean()),
        f"{other_name}_mean": float(
            paired[other_name].mean()
        ),
        "mean_signed_difference_C2_minus_other": float(
            np.mean(difference)
        ),
        "absolute_mean_difference": float(
            abs(np.mean(difference))
        ),
    }

In [ ]:
# Runs C2 against every other configuration whose fold-metric CSV can be found under experiments/.
COMPARISON_METRICS = ["f1_macro", "auroc_macro_ovr", "brier_score", "ece_15", "nll"]
OTHER_CONFIGS = ["C1", "B1", "B2", "B3", "B4"]

# With n = 5 pairs the smallest possible two-sided exact p-value is 2/2**5 = 0.0625, so alpha = 0.05 can never be
# reached. Report the effect size (mean difference) first and discuss
print("Smallest possible two-sided p with 5 pairs:", 2 / 2**5)

def find_fold_metrics_csv(prefix):
    hits = sorted((PROJECT_ROOT / "experiments").glob(f"{prefix}_*/metrics/*fold_metrics*.csv"))
    return hits[0] if hits else None

comparison_rows = []
c2_csv = METRICS_ROOT / "C2_fold_metrics.csv"

if c2_csv.exists():
    for other in OTHER_CONFIGS:
        other_csv = find_fold_metrics_csv(other)
        if other_csv is None:
            print(f"{other}: fold-metric CSV not found, skipped")
            continue
        for metric in COMPARISON_METRICS:
            try:
                row = paired_wilcoxon_comparison(c2_csv, other_csv, metric, other_name=other)
                row["compared_with"] = other
                comparison_rows.append(row)
            except Exception as exc:
                print(f"{other} / {metric}: {exc}")

if comparison_rows:
    wilcoxon_table = pd.DataFrame(comparison_rows)
    wilcoxon_table.to_csv(REPORT_ROOT / "C2_wilcoxon_comparisons.csv", index=False)
    display(wilcoxon_table)

Smallest possible two-sided p with 5 pairs: 0.0625
C1: fold-metric CSV not found, skipped
B1 / f1_macro: "['f1_macro'] not in index"
B1 / auroc_macro_ovr: "['auroc_macro_ovr'] not in index"
B1 / brier_score: Formal comparison requires 5 paired folds; found 4
B1 / ece_15: Formal comparison requires 5 paired folds; found 4
B1 / nll: Formal comparison requires 5 paired folds; found 4
B2 / f1_macro: "['f1_macro'] not in index"
B2 / auroc_macro_ovr: "['auroc_macro_ovr'] not in index"
B2 / brier_score: Formal comparison requires 5 paired folds; found 4
B2 / ece_15: Formal comparison requires 5 paired folds; found 4
B2 / nll: Formal comparison requires 5 paired folds; found 4
B3 / f1_macro: Formal comparison requires 5 paired folds; found 4
B3 / auroc_macro_ovr: Formal comparison requires 5 paired folds; found 4
B3 / brier_score: Formal comparison requires 5 paired folds; found 4
B3 / ece_15: Formal comparison requires 5 paired folds; found 4
B3 / nll: Formal comparison requires 5 paired fold

23. Comparison-ready export and manifest



In [ ]:
def build_comparison_ready_table():
    path = METRICS_ROOT / "C2_fold_metrics.csv"

    if not path.exists():
        print("Run C2 first.")
        return None

    df = pd.read_csv(path)

    wanted = [
        "fold",
        "f1_macro",
        "auroc_macro_ovr",
        "brier_score",
        "ece_15",
        "nll",
        "rejection_auc_vacuity",
        "single_pass_latency_ms_per_image",
        "single_pass_flops",
        "total_parameters",
        "alignment_mse",
        "entropy_variance_pearson_r",
        "mean_vacuity",
        "mean_aleatoric_entropy",
        "mean_dirichlet_strength",
        "lambda_max",
        "T_warmup",
    ]

    available = [
        c for c in wanted
        if c in df.columns
    ]

    output = df[available].copy()

    output.insert(1, "config_id", "C2")
    output.insert(2, "backbone", "MedMamba-T")
    output.insert(3, "uq_method", "EDL + HAAL")
    output.insert(4, "loss", "L_ECE + lambda(t)L_Align")

    output.to_csv(
        REPORT_ROOT / "C2_comparison_ready.csv",
        index=False,
    )

    return output

if RUN_MAIN_5FOLD:
    display(build_comparison_ready_table())

manifest = {
    "configuration": "C2",
    "backbone": "MedMamba-T",
    "official_medmamba_source": OFFICIAL_MEDMAMBA_URL,
    "depths": CONFIG["medmamba_depths"],
    "dims": CONFIG["medmamba_dims"],
    "uq_method": "EDL + HAAL",
    "main_loss": "L_ECE + lambda(t) * L_Align",
    "KL_in_main_C2": False,
    "prior_C": CONFIG["prior_weight_C"],
    "lambda_max": CONFIG["lambda_max"],
    "T_warmup": CONFIG["T_warmup"],
    "haal_source": CONFIG["haal_source"],
    "variance_target_definition": (
        "population variance of >=3 radiologist malignancy scores / 4"
    ),
    "variance_target_source": variance_source,
    "author_weight_path": str(MEDMAMBA_WEIGHTS_PATH),
    "author_weight_available": MEDMAMBA_WEIGHTS_PATH.exists(),
    "pretraining_policy": (
        "Use author-released MedMamba weights when available; "
        "otherwise train from scratch and report it."
    ),
    "primary_esm_hook": PRIMARY_ESM_HOOK,
    "esm_hook_ablation": ["pre", "post", "internal"],
    "patients_used": int(metadata.patient_id.nunique()),
    "patches_used": int(len(metadata)),
    "pilot_max_patients": CONFIG["pilot_max_patients"],
    "folds": CONFIG["n_splits"],
    "seed": CONFIG["seed"],
    "medmamba_source_sha256": MEDMAMBA_SHA256,
    "fold_source": outer_method,
    "fold_ids": FOLD_IDS,
    "zero_variance_excluded_from_alignment": True,
    "scope": "2D single-slice LIDC-IDRI patches",
    "external_validation": False,
    "clinical_deployment_claim": True, #False
}

with open(
    REPORT_ROOT / "C2_experiment_manifest.json",
    "w",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
    )

print(json.dumps(manifest, indent=2))



# results summary and notes, same set of files as the other configurations
if RUN_MAIN_5FOLD and (METRICS_ROOT / "C2_5fold_mean_std.csv").exists():
    summary_out = pd.read_csv(METRICS_ROOT / "C2_5fold_mean_std.csv")
    summary_out.to_csv(REPORT_ROOT / "C2_results_summary.csv", index=False)
    summary_out.to_json(REPORT_ROOT / "C2_results_summary.json", orient="records", indent=2)

    (REPORT_ROOT / "C2_experiment_notes.md").write_text(
        "# C2 MedMamba-T + EDL + HAAL\n\n"
        f"- HAAL settings: {CONFIG['haal_source']}\n"
        f"- C = {CONFIG['prior_weight_C']}, lambda_max = {CONFIG['lambda_max']}, T_warmup = {CONFIG['T_warmup']}\n"
        f"- Folds: {outer_method}, ids {FOLD_IDS}\n"
        f"- Pretraining: see pretraining_status column in metrics/C2_fold_metrics.csv\n"
        f"- Patients: {metadata.patient_id.nunique()}, patches: {len(metadata)}\n"
        "- Zero-variance nodules excluded from L_Align only.\n"
        "- Scope: 2D single-slice, no external validation, research use only.\n"
    )
    print("Saved results summary and notes to", REPORT_ROOT)

,fold,config_id,backbone,uq_method,loss,f1_macro,auroc_macro_ovr,brier_score,ece_15,nll,...,single_pass_latency_ms_per_image,single_pass_flops,total_parameters,alignment_mse,entropy_variance_pearson_r,mean_vacuity,mean_aleatoric_entropy,mean_dirichlet_strength,lambda_max,T_warmup
0,1,C2,MedMamba-T,EDL + HAAL,L_ECE + lambda(t)L_Align,0.172840,0.643384,0.640648,0.138748,1.022071,...,23.090374,2.552186e+09,14454915,0.851783,NaN,0.288271,1.013426,10.415929,5.0,0
1,2,C2,MedMamba-T,EDL + HAAL,L_ECE + lambda(t)L_Align,0.550000,0.575000,0.590314,0.154605,0.968600,...,23.339744,2.552186e+09,14454915,0.930453,NaN,0.218409,1.058500,13.739175,5.0,0
2,3,C2,MedMamba-T,EDL + HAAL,L_ECE + lambda(t)L_Align,0.133333,0.512193,0.846303,0.316319,1.454979,...,23.075128,2.552186e+09,14454915,0.723952,NaN,0.176723,0.930466,17.033152,5.0,0
3,4,C2,MedMamba-T,EDL + HAAL,L_ECE + lambda(t)L_Align,0.242424,0.332906,0.612785,0.338867,0.973978,...,23.138017,2.552186e+09,14454915,0.773956,NaN,0.180753,0.957778,16.611395,5.0,0
4,5,C2,MedMamba-T,EDL + HAAL,L_ECE + lambda(t)L_Align,0.235294,0.532524,0.644975,0.083792,1.079975,...,22.686053,2.552186e+09,14454915,0.842294,NaN,0.411485,1.008251,7.291503,5.0,0


{
  "configuration": "C2",
  "backbone": "MedMamba-T",
  "official_medmamba_source": "https://raw.githubusercontent.com/YubiaoYue/MedMamba/main/MedMamba.py",
  "depths": [
    2,
    2,
    4,
    2
  ],
  "dims": [
    96,
    192,
    384,
    768
  ],
  "uq_method": "EDL + HAAL",
  "main_loss": "L_ECE + lambda(t) * L_Align",
  "KL_in_main_C2": false,
  "prior_C": 1.0,
  "lambda_max": 5.0,
  "T_warmup": 0,
  "haal_source": "inherited_from:/content/drive/MyDrive/LIDC_Thesis/experiments/C1_ConvNeXtV2T_EDL_HAAL/configs/C1_config.json",
  "variance_target_definition": "population variance of >=3 radiologist malignancy scores / 4",
  "variance_target_source": "not_required_for_reader_distribution",
  "author_weight_path": "/content/drive/MyDrive/LIDC_Thesis/models/MedMamba/weights/medmamba_t.pth",
  "author_weight_available": false,
  "pretraining_policy": "Use author-released MedMamba weights when available; otherwise train from scratch and report it.",
  "primary_esm_hook": "post",
  "e